# Sentinel-2 crop/non-crop mapping: four-model benchmark

Research notebook containing DINOv2, 3D UNet++, TemporalViT, and UNet–ConvLSTM implementations.

**Research release note:** This notebook is provided for inspection and reproduction; it has not been independently rerun or verified in a clean environment. Review the README and manuscript consistency notes before citing results.


In [ ]:
# Full notebook cell: include all regions and years automatically
# Paste into a Jupyter cell and run.
# Requirements: transformers, rasterio, torch, torchvision, sklearn, matplotlib, pandas

import os, random
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import rasterio
import matplotlib.pyplot as plt
from transformers import Dinov2Model, AutoImageProcessor
from sklearn.metrics import confusion_matrix, cohen_kappa_score, accuracy_score, precision_recall_fscore_support
import pandas as pd

# ---------------- USER CONFIG ----------------
DATA_ROOT = Path('/kaggle/input/sentinel2-crop-mapping')  # change if needed
MAX_TILES_PER_REGION = 30     # sample this many tileids per region (keeps run small)
MAX_IMAGES_PER_TILE = 6       # timestamps per tile to keep
IMAGE_SIZE = 128              # input & mask resolution
BATCH_SIZE = 8
EPOCHS = 6
NUM_WORKERS = 0               # use 0 while debugging; raise later for speed
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BACKBONE = 'facebook/dinov2-small'
OUT_DIR = Path('output'); OUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 42
FREEZE_EPOCHS = 2             # freeze backbone for first N epochs
# ----------------------------------------------

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# load processor for mean/std (if fails, we'll fallback)
try:
    processor = AutoImageProcessor.from_pretrained(BACKBONE)
except Exception as e:
    print("AutoImageProcessor load failed, proceeding with fallback normalization. Error:", e)
    processor = None

# ---------------- detect regions and years ----------------
region_dirs = []
for d in DATA_ROOT.iterdir():
    if not d.is_dir():
        continue
    # consider it a region if it contains any folder that starts with 'data' and digits afterwards
    if any(child.is_dir() and child.name.startswith('data') for child in d.iterdir()):
        region_dirs.append(d.name)
region_dirs = sorted(region_dirs)
if len(region_dirs) == 0:
    raise RuntimeError(f"No region directories with 'dataYYYY' found under {DATA_ROOT}. Check dataset path.")

print("Detected regions:", region_dirs)

# map regions -> available years (int)
region_years = {}
for r in region_dirs:
    rpath = DATA_ROOT / r
    years = []
    for child in rpath.iterdir():
        if child.is_dir() and child.name.startswith('data'):
            # try parse year
            try:
                year_int = int(child.name.replace('data',''))
                years.append(year_int)
            except:
                continue
    region_years[r] = sorted(years)
print("Detected region years (sample):", {k: v for k, v in list(region_years.items())[:10]})

# ---------------- build sample list across all regions & years ----------------
sample_images = []  # list of tuples (region, year, tileid, filename)
for region in region_dirs:
    years = region_years.get(region, [])
    # collect tileids for this region across all years
    tileids = set()
    for y in years:
        base = DATA_ROOT / region / f"data{y}"
        if not base.exists():
            continue
        for t in base.iterdir():
            if t.is_dir():
                tileids.add(t.name)
    tileids = sorted(tileids)
    # sample up to MAX_TILES_PER_REGION tileids (or take all if fewer)
    chosen_tileids = random.sample(tileids, min(MAX_TILES_PER_REGION, len(tileids)))
    for tileid in chosen_tileids:
        # for each selected tileid, gather up to MAX_IMAGES_PER_TILE timestamps across years (prefer earlier years first)
        collected = []
        for y in years:
            tdir = DATA_ROOT / region / f"data{y}" / tileid
            if not tdir.exists():
                continue
            tifs = sorted([p.name for p in tdir.glob("*.tif")])
            for tname in tifs:
                if len(collected) < MAX_IMAGES_PER_TILE:
                    collected.append((region, y, tileid, tname))
        # if across years we still didn't fill MAX_IMAGES_PER_TILE, try again with any tif under any year
        if len(collected) == 0:
            # try to find any tifs in any year dir for this tile
            for y in years:
                tdir = DATA_ROOT / region / f"data{y}" / tileid
                if tdir.exists():
                    tifs = sorted([p.name for p in tdir.glob("*.tif")])
                    for tname in tifs[:MAX_IMAGES_PER_TILE]:
                        collected.append((region, y, tileid, tname))
        sample_images += collected

print("Total sampled region-year-tile-image entries:", len(sample_images))
if len(sample_images) == 0:
    raise RuntimeError("No sample images discovered. Check dataset structure and paths.")

# ---------------- helper: read image for region/year/tile/file ----------------
def read_image_for_entry(region_root: Path, region: str, year: int, tileid: str, fname: str, target_size=IMAGE_SIZE):
    """
    Returns (PIL_resized, numpy_resized_bands) or (None,None)
    """
    p = region_root / region / f"data{year}" / tileid / fname
    if not p.exists():
        # fallback: search in region/data*/tileid for matching filename (some datasets have different placements)
        candidates = []
        region_dir = region_root / region
        for child in region_dir.iterdir():
            if child.is_dir() and child.name.startswith('data'):
                t = child / tileid / fname
                if t.exists():
                    candidates.append(t)
        if len(candidates) == 0:
            return None, None
        p = candidates[0]
    try:
        with rasterio.open(p) as src:
            arr = src.read(out_dtype='float32')  # (C, H, W)
            if arr.shape[0] >= 3:
                img_arr = arr[:3]
            else:
                img_arr = np.vstack([arr] * (3 // arr.shape[0] + 1))[:3]
            img_arr = img_arr.astype('float32')
            img_arr = (img_arr - img_arr.min()) / (img_arr.max() - img_arr.min() + 1e-6)
            img_hwc = np.moveaxis(img_arr, 0, -1)  # H,W,C
            pil = Image.fromarray((np.clip(img_hwc,0,1) * 255).astype('uint8'))
            pil = pil.resize((target_size, target_size), resample=Image.BILINEAR)
            np_resized = np.asarray(pil).astype('float32') / 255.0
            return pil, np_resized
    except Exception as e:
        # return None, None silently (could print e for debug)
        return None, None

# ---------------- Dataset that takes (region, year, tileid, fname) entries ----------------
class SentinelAllRegionsDataset(Dataset):
    def __init__(self, samples, data_root, image_size=IMAGE_SIZE, processor=None):
        self.samples = samples
        self.data_root = data_root
        self.image_size = image_size
        self.processor = processor
        # quick estimate of class balance
        pos = 0
        tot = 0
        for i, (region, year, tileid, fname) in enumerate(self.samples[:min(len(self.samples), 200)]):
            pil, arr = read_image_for_entry(self.data_root, region, year, tileid, fname, target_size=self.image_size)
            if arr is None:
                continue
            mask = (arr[...,0] > arr[...,0].mean()).astype('int64')
            pos += mask.sum()
            tot += mask.size
        self.pos = int(pos)
        self.tot = int(tot) if tot>0 else 1
        self.pos_frac = self.pos/self.tot if self.tot>0 else 0.0
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        region, year, tileid, fname = self.samples[idx]
        pil, arr = read_image_for_entry(self.data_root, region, year, tileid, fname, target_size=self.image_size)
        if pil is None:
            pixel_values = torch.zeros((3, self.image_size, self.image_size), dtype=torch.float32)
            mask = torch.zeros((self.image_size, self.image_size), dtype=torch.long)
            return pixel_values, mask
        tensor = transforms.ToTensor()(pil)  # 0..1
        # normalize using processor mean/std if available
        if self.processor is not None and hasattr(self.processor, "image_mean") and hasattr(self.processor, "image_std"):
            mean = torch.tensor(self.processor.image_mean, dtype=torch.float32).view(3,1,1)
            std  = torch.tensor(self.processor.image_std, dtype=torch.float32).view(3,1,1)
            tensor = (tensor - mean) / std
        else:
            tensor = (tensor - 0.5) / 0.5
        if arr is None:
            mask_np = np.zeros((self.image_size, self.image_size), dtype='int64')
        else:
            mask_np = (arr[...,0] > arr[...,0].mean()).astype('int64')
        mask_t = torch.from_numpy(mask_np).long()
        return tensor.float(), mask_t

# ---------------- split into train/val and dataloaders ----------------
random.shuffle(sample_images)
split_idx = int(0.8 * len(sample_images))
train_samples = sample_images[:split_idx]
val_samples = sample_images[split_idx: split_idx + 400]  # limit val size to 400 entries

train_ds = SentinelAllRegionsDataset(train_samples, DATA_ROOT, image_size=IMAGE_SIZE, processor=processor)
val_ds   = SentinelAllRegionsDataset(val_samples, DATA_ROOT, image_size=IMAGE_SIZE, processor=processor)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("Train/Val sample counts:", len(train_ds), len(val_ds))
print("Estimated positive pixel count (train):", train_ds.pos, "of", train_ds.tot, "=> frac:", train_ds.pos_frac)

# ---------------- model ----------------
class SimpleDinov2Seg(nn.Module):
    def __init__(self, backbone_name=BACKBONE, num_classes=2):
        super().__init__()
        self.backbone = Dinov2Model.from_pretrained(backbone_name)
        hidden_dim = self.backbone.config.hidden_size
        patch_size = getattr(self.backbone.config, 'patch_size', 16)
        self.patch_size = patch_size
        self.hidden_dim = hidden_dim
        self.conv1 = nn.Conv2d(hidden_dim, hidden_dim//2, kernel_size=3, padding=1)
        self.upsample = nn.Sequential(
            nn.ConvTranspose2d(hidden_dim//2, hidden_dim//4, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(hidden_dim//4, hidden_dim//8, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
        )
        self.final_conv = nn.Conv2d(hidden_dim//8, num_classes, kernel_size=1)
    def forward(self, x):
        out = self.backbone(pixel_values=x)['last_hidden_state']  # B, seq_len, hidden
        seq_len = out.shape[1]
        expected_seq = (x.shape[2] // self.patch_size) * (x.shape[3] // self.patch_size)
        if seq_len == expected_seq + 1:
            out = out[:,1:,:]
        Hpatch = x.shape[2] // self.patch_size
        Wpatch = x.shape[3] // self.patch_size
        out = out.permute(0,2,1).contiguous().view(x.shape[0], self.hidden_dim, Hpatch, Wpatch)
        out = self.conv1(out)
        out = self.upsample(out)
        out = self.final_conv(out)
        out = nn.functional.interpolate(out, size=(x.shape[2], x.shape[3]), mode='bilinear', align_corners=False)
        return out

model = SimpleDinov2Seg().to(DEVICE)

# ---------------- loss + optimizer ----------------
pos = train_ds.pos
tot = train_ds.tot if train_ds.tot>0 else 1
neg = tot - pos
if pos == 0:
    class_weights = torch.tensor([1.0, 1.0], dtype=torch.float32)
else:
    w0 = 1.0
    w1 = float(max(1.0, neg / (pos + 1e-6)))
    class_weights = torch.tensor([w0, w1], dtype=torch.float32)
print("Class weights:", class_weights.numpy())

# freeze backbone initially
for p in model.backbone.parameters():
    p.requires_grad = False
decoder_params = [p for n,p in model.named_parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(decoder_params, lr=5e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))

# optional sanity check
if len(train_loader) > 0:
    imgs, masks = next(iter(train_loader))
    print("Sample batch shapes imgs", imgs.shape, "masks", masks.shape)
    assert imgs.shape[2] == IMAGE_SIZE and imgs.shape[3] == IMAGE_SIZE, "Image spatial size mismatch"
    assert masks.shape[1] == IMAGE_SIZE and masks.shape[2] == IMAGE_SIZE, "Mask spatial size mismatch"

# ---------------- training loop ----------------
metrics_rows = []
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    total_samples = 0
    for imgs, masks in train_loader:
        imgs = imgs.to(DEVICE)
        masks = masks.to(DEVICE)
        outputs = model(imgs)      # B,C,H,W
        loss = criterion(outputs, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        total_samples += imgs.size(0)
    avg_loss = total_loss / max(1, total_samples)
    print(f"Epoch {epoch+1}/{EPOCHS} train_loss={avg_loss:.4f}")

    # unfreeze backbone after FREEZE_EPOCHS
    if epoch+1 == FREEZE_EPOCHS:
        print("Unfreezing backbone for fine-tuning...")
        for p in model.backbone.parameters():
            p.requires_grad = True
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-5)

    # validation metrics
    model.eval()
    conf = np.zeros((2,2), dtype=np.int64)
    all_preds = []
    all_gts = []
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.to(DEVICE)
            masks = masks.to(DEVICE)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1).cpu().numpy().ravel()
            gts = masks.cpu().numpy().ravel()
            cm = confusion_matrix(gts, preds, labels=[0,1])
            conf += cm
            all_preds.append(preds)
            all_gts.append(gts)
    if len(all_preds) > 0:
        all_preds = np.concatenate(all_preds)
        all_gts = np.concatenate(all_gts)
    else:
        all_preds = np.array([])
        all_gts = np.array([])

    OA = accuracy_score(all_gts, all_preds) if all_gts.size>0 else 0.0
    Kappa = cohen_kappa_score(all_gts, all_preds) if all_gts.size>0 else 0.0
    iou_per_class = []
    for c in range(2):
        tp = conf[c,c]
        fp = conf[:,c].sum() - tp
        fn = conf[c,:].sum() - tp
        denom = tp + fp + fn
        iou = tp / denom if denom>0 else 0.0
        iou_per_class.append(iou)
    mIoU = float(np.mean(iou_per_class))
    p, r, f1, sup = precision_recall_fscore_support(all_gts, all_preds, labels=[0,1], zero_division=0)
    wP = (p * sup).sum() / max(1, sup.sum())
    wR = (r * sup).sum() / max(1, sup.sum())
    wF1= (f1 * sup).sum() / max(1, sup.sum())

    print(f" Val OA={OA:.4f} Kappa={Kappa:.4f} mIoU={mIoU:.4f} wP={wP:.4f} wR={wR:.4f} wF1={wF1:.4f}")
    metrics_rows.append({'epoch':epoch+1, 'loss':avg_loss, 'OA':OA, 'Kappa':Kappa, 'mIoU':mIoU, 'wP':wP, 'wR':wR, 'wF1':wF1,
                         'conf00':int(conf[0,0]), 'conf01':int(conf[0,1]), 'conf10':int(conf[1,0]), 'conf11':int(conf[1,1])})
    torch.save({'epoch': epoch+1, 'model_state': model.state_dict(), 'optimizer': optimizer.state_dict()}, OUT_DIR / f'model_epoch{epoch+1}.pth')

pd.DataFrame(metrics_rows).to_csv(OUT_DIR / 'metrics.csv', index=False)
print("Saved metrics.csv")

# ---------------- visualization ----------------
model.eval()
try:
    imgs, masks = next(iter(val_loader))
except StopIteration:
    print("No validation images to visualize.")
else:
    with torch.no_grad():
        imgs_device = imgs.to(DEVICE)
        outs = model(imgs_device)
        preds = outs.argmax(dim=1).cpu().numpy()
    nvis = min(6, imgs.shape[0])
    fig, axs = plt.subplots(nvis, 3, figsize=(9, 3*nvis))
    for i in range(nvis):
        im = imgs[i].cpu().permute(1,2,0).numpy()
        # undo normalization for display if we used processor stats
        if processor is not None and hasattr(processor, "image_mean") and hasattr(processor, "image_std"):
            mean = np.array(processor.image_mean).reshape(1,1,3)
            std  = np.array(processor.image_std).reshape(1,1,3)
            im = (im * std) + mean
        im = np.clip(im, 0, 1)
        axs[i,0].imshow(im)
        axs[i,0].set_title('input')
        axs[i,1].imshow(masks[i].cpu().numpy(), cmap='tab20', vmin=0, vmax=1)
        axs[i,1].set_title('gt')
        axs[i,2].imshow(preds[i], cmap='tab20', vmin=0, vmax=1)
        axs[i,2].set_title('pred')
        for j in range(3):
            axs[i,j].axis('off')
    plt.tight_layout()
    out_vis = OUT_DIR / 'val_predictions_allregions.png'
    plt.savefig(out_vis, dpi=200)
    print("Saved visualization to", out_vis)

print("Done — used regions:", region_dirs, "and years per region (sample):", {r: region_years[r] for r in region_dirs})

In [ ]:
# Full evaluation + EDA cell
# Requirements: torch, torchvision, transformers, rasterio (recommended) or tifffile, sklearn, matplotlib, seaborn, pandas, tqdm
import os, io
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix, accuracy_score, cohen_kappa_score, precision_recall_fscore_support, roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

# Optional geotiff readers
try:
    import rasterio
except Exception:
    rasterio = None
try:
    import tifffile
except Exception:
    tifffile = None

# transformers
try:
    from transformers import Dinov2Model, AutoImageProcessor
    HF_AVAILABLE = True
except Exception:
    Dinov2Model = None
    AutoImageProcessor = None
    HF_AVAILABLE = False

# ---------------- CONFIG ----------------
DATA_ROOT = Path("/kaggle/input/sentinel2-crop-mapping")   # change if needed
MODEL_PATH = Path("output/model_epoch6.pth")               # change if needed
BACKBONE = "facebook/dinov2-small"
IMAGE_SIZE = 128      # must match model input
BATCH_SIZE = 8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_SAMPLES = 300     # how many tiles to evaluate (set to None to evaluate all found)
OUTPUT_DIR = Path("output_eval"); OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
RANDOM_SEED = 42
THRESHOLD = 0.5       # default probability threshold for binary mask

random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)

# ---------------- Helpers: dataset discovery & readers ----------------
def discover_tif_files(data_root: Path, max_samples=None):
    """Return list of tif file paths discovered under data{year}/tileid folders across regions"""
    files = []
    for region in sorted([d for d in data_root.iterdir() if d.is_dir()]):
        # look for dataYYYY subdirs
        for data_dir in sorted([p for p in region.iterdir() if p.is_dir() and p.name.startswith("data")]):
            for tile in sorted([t for t in data_dir.iterdir() if t.is_dir()]):
                # collect all .tif files in tile
                tifs = sorted(list(tile.glob("*.tif")))
                for f in tifs:
                    files.append(f)
    if max_samples is not None and len(files) > max_samples:
        files = random.sample(files, max_samples)
    return files

def read_tif_as_rgb(path_or_bytes, image_size=IMAGE_SIZE, percentile_clip=True, p_low=2, p_high=98):
    """
    Read a GeoTIFF path (Path or file-like bytes) and return PIL-like numpy HxWx3 scaled to [0,1]
    Falls back to tifffile if rasterio not available. Returns (img_array_float_HWC, profile_or_None)
    """
    profile = None
    if rasterio is not None:
        # path_or_bytes can be a Path
        if isinstance(path_or_bytes, (str, Path)):
            p = str(path_or_bytes)
            with rasterio.open(p) as src:
                arr = src.read(out_dtype='float32')  # C,H,W
                profile = src.profile.copy()
        else:
            # file-like bytes
            mem = io.BytesIO(path_or_bytes)
            with rasterio.open(mem) as src:
                arr = src.read(out_dtype='float32')
                profile = src.profile.copy()
    elif tifffile is not None:
        if isinstance(path_or_bytes, (str, Path)):
            arr = tifffile.imread(str(path_or_bytes)).astype('float32')
        else:
            arr = tifffile.imread(path_or_bytes).astype('float32')
        # tifffile returns HWC or CHW; normalize to CHW
        if arr.ndim == 2:
            arr = arr[np.newaxis, :, :]
        elif arr.ndim == 3 and arr.shape[2] <= 4 and arr.shape[0] != arr.shape[2]:
            # likely HWC
            if arr.shape[2] <= 4:
                arr = np.moveaxis(arr, -1, 0)
    else:
        raise RuntimeError("Neither rasterio nor tifffile available to read GeoTIFFs.")

    # now arr is C,H,W
    if arr.shape[0] >= 3:
        rgb = arr[:3]
    else:
        # replicate channels
        reps = int(np.ceil(3.0 / max(1, arr.shape[0])))
        rgb = np.tile(arr, (reps,1,1))[:3]

    # percentile stretch to [0,1] (robust to outliers)
    if percentile_clip:
        lo = np.nanpercentile(rgb, p_low)
        hi = np.nanpercentile(rgb, p_high)
        rgb = np.clip((rgb - lo) / (hi - lo + 1e-9), 0, 1)
    else:
        mn, mx = rgb.min(), rgb.max()
        rgb = (rgb - mn) / (mx - mn + 1e-9)

    img = np.moveaxis(rgb, 0, -1)  # H,W,3 in [0,1]
    # resize to IMAGE_SIZE
    from PIL import Image
    pil = Image.fromarray((img*255).astype('uint8')).convert('RGB').resize((image_size, image_size), resample=Image.BILINEAR)
    img_resized = np.array(pil).astype('float32') / 255.0
    return img_resized, profile

# ---------------- Model loader (robust) ----------------
class SimpleDinov2Seg(nn.Module):
    def __init__(self, backbone_name=BACKBONE, num_classes=2):
        super().__init__()
        # requires transformers installed with Dinov2Model
        self.backbone = Dinov2Model.from_pretrained(backbone_name)
        hidden_dim = self.backbone.config.hidden_size
        patch_size = getattr(self.backbone.config, 'patch_size', 16)
        self.patch_size = patch_size
        self.hidden_dim = hidden_dim
        self.conv1 = nn.Conv2d(hidden_dim, hidden_dim//2, 3, padding=1)
        self.upsample = nn.Sequential(
            nn.ConvTranspose2d(hidden_dim//2, hidden_dim//4, 2, 2),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(hidden_dim//4, hidden_dim//8, 2, 2),
            nn.ReLU(inplace=True),
        )
        self.final_conv = nn.Conv2d(hidden_dim//8, num_classes, 1)

    def forward(self, x):
        out = self.backbone(pixel_values=x)['last_hidden_state']
        seq_len = out.shape[1]
        expected_seq = (x.shape[2] // self.patch_size) * (x.shape[3] // self.patch_size)
        if seq_len == expected_seq + 1:
            out = out[:,1:,:]
        Hpatch = x.shape[2] // self.patch_size
        Wpatch = x.shape[3] // self.patch_size
        out = out.permute(0,2,1).contiguous().view(x.shape[0], self.hidden_dim, Hpatch, Wpatch)
        out = self.conv1(out)
        out = self.upsample(out)
        out = self.final_conv(out)
        out = nn.functional.interpolate(out, size=(x.shape[2], x.shape[3]), mode='bilinear', align_corners=False)
        return out

def load_model(checkpoint_path: Path, device=DEVICE):
    print("Loading model from", checkpoint_path)
    model = SimpleDinov2Seg()
    ckpt = torch.load(str(checkpoint_path), map_location='cpu')
    if isinstance(ckpt, dict):
        # handle common wrappers
        state = ckpt.get('model_state', ckpt.get('state_dict', ckpt))
    else:
        state = ckpt
    model.load_state_dict(state, strict=False)
    model.to(device)
    model.eval()
    return model

# ---------------- Preprocess function (matches earlier pipeline) ----------------
def preprocess_img_for_model(img_hwc, processor=None):
    """img_hwc in [0,1], HWC. returns 1xCxHxW torch tensor on DEVICE"""
    x = torch.tensor(img_hwc).permute(2,0,1).unsqueeze(0).float()  # 1,C,H,W
    if processor is not None and hasattr(processor, "image_mean") and hasattr(processor, "image_std"):
        mean = torch.tensor(processor.image_mean, dtype=torch.float32).view(1,3,1,1)
        std  = torch.tensor(processor.image_std, dtype=torch.float32).view(1,3,1,1)
        x = (x - mean) / (std + 1e-9)
    else:
        x = (x - 0.5) / 0.5
    return x.to(DEVICE)

# ---------------- Mask extraction (if no GT provided) ----------------
def extract_mask_from_arr(arr_hwc):
    """
    If real GT masks exist, replace this function.
    Current heuristic (used earlier): class 1 where band0 > band0.mean()
    arr_hwc: H,W,3 in [0,1]. Return HxW int mask {0,1}
    """
    ch0 = arr_hwc[...,0]
    threshold = ch0.mean()
    mask = (ch0 > threshold).astype(np.uint8)
    return mask

# ---------------- EVALUATION loop ----------------
# Discover files
tif_files = discover_tif_files(DATA_ROOT, max_samples=MAX_SAMPLES)
print(f"Discovered {len(tif_files)} tif files (sample)")

# Load model + processor (if available)
processor = None
if HF_AVAILABLE and AutoImageProcessor is not None:
    try:
        processor = AutoImageProcessor.from_pretrained(BACKBONE)
        print("Loaded AutoImageProcessor")
    except Exception as e:
        print("AutoImageProcessor load failed:", e)
model = load_model(MODEL_PATH, DEVICE)

# Containers for per-sample metrics
rows = []
all_preds = []
all_gts = []
all_probs = []  # class-1 probability flattened
conf_total = np.zeros((2,2), dtype=np.int64)

# Evaluate in batches
from math import ceil
n = len(tif_files)
num_batches = ceil(n / BATCH_SIZE)
print("Batches:", num_batches)

for i in tqdm(range(num_batches), desc="Evaluating"):
    batch_files = tif_files[i*BATCH_SIZE: (i+1)*BATCH_SIZE]
    imgs = []
    profiles = []
    orig_shapes = []
    filenames = []
    for f in batch_files:
        try:
            img, profile = read_tif_as_rgb(f, image_size=IMAGE_SIZE, percentile_clip=True)
        except Exception as e:
            # skip problematic files but log
            print("Read failed for", f, e)
            continue
        imgs.append(img)
        profiles.append(profile)
        orig_shapes.append(img.shape[:2])
        filenames.append(str(f.relative_to(DATA_ROOT.parent)) if DATA_ROOT.parent in f.parents else str(f))
    if len(imgs) == 0:
        continue

    # preprocess to tensor
    tensors = []
    for img in imgs:
        t = preprocess_img_for_model(img, processor=processor)
        tensors.append(t)
    batch_x = torch.cat(tensors, dim=0)  # B,C,H,W

    with torch.no_grad():
        logits = model(batch_x)  # B,C,H,W
        probs = torch.softmax(logits, dim=1).cpu().numpy()  # B,C,H,W
        # class predictions
        preds = probs.argmax(axis=1)  # B,H,W
        class1_probs = probs[:,1,:,:] if probs.shape[1] > 1 else probs[:,0,:,:]

    # compute per-image metrics
    for idx in range(len(imgs)):
        img_hwc = imgs[idx]
        gt_mask = extract_mask_from_arr(img_hwc)  # H,W
        pred_mask = preds[idx].astype(np.uint8)
        prob_map = class1_probs[idx]

        # flatten
        gts = gt_mask.ravel()
        prd = pred_mask.ravel()
        probs_flat = prob_map.ravel()

        # confusion
        cm = confusion_matrix(gts, prd, labels=[0,1])
        conf_total += cm

        # metrics
        OA = accuracy_score(gts, prd)
        try:
            Kappa = cohen_kappa_score(gts, prd)
        except Exception:
            Kappa = float('nan')
        # per-class IoU
        iou_per_class = []
        for c in [0,1]:
            tp = int(cm[c,c])
            fp = int(cm[:,c].sum()) - tp
            fn = int(cm[c,:].sum()) - tp
            denom = tp + fp + fn
            iou = (tp/denom) if denom>0 else 0.0
            iou_per_class.append(iou)
        mIoU = float(np.mean(iou_per_class))
        p, r, f1, sup = precision_recall_fscore_support(gts, prd, labels=[0,1], zero_division=0)
        # weighted versions
        support = np.array(sup)
        total_sup = support.sum() if support.sum() > 0 else 1
        wP = float((p * support).sum() / total_sup)
        wR = float((r * support).sum() / total_sup)
        wF1 = float((f1 * support).sum() / total_sup)
        # ROC AUC & AP for class1 (if both classes present)
        try:
            if len(np.unique(gts)) > 1:
                AUROC = roc_auc_score(gts, probs_flat)
                AP = average_precision_score(gts, probs_flat)
            else:
                AUROC = float('nan'); AP = float('nan')
        except Exception:
            AUROC = float('nan'); AP = float('nan')

        row = {
            "file": filenames[idx],
            "OA": OA, "Kappa": Kappa, "mIoU": mIoU,
            "iou_c0": iou_per_class[0], "iou_c1": iou_per_class[1],
            "wP": wP, "wR": wR, "wF1": wF1,
            "AUROC": AUROC, "AP": AP,
            "conf00": int(cm[0,0]), "conf01": int(cm[0,1]), "conf10": int(cm[1,0]), "conf11": int(cm[1,1]),
            "gt_pos_pixels": int(gts.sum()), "pred_pos_pixels": int(prd.sum()),
        }
        rows.append(row)
        all_preds.append(prd)
        all_gts.append(gts)
        all_probs.append(probs_flat)

# ---------------- Aggregate & save metrics ----------------
df = pd.DataFrame(rows)
if df.shape[0] == 0:
    raise RuntimeError("No evaluation rows were produced. Check dataset discovery and reading.")
df.to_csv(OUTPUT_DIR / "metrics_full.csv", index=False)
print("Saved per-image metrics to", OUTPUT_DIR / "metrics_full.csv")

# aggregated summary (best or mean)
summary = {
    "n_samples": int(df.shape[0]),
    "OA_mean": float(df["OA"].mean()),
    "Kappa_mean": float(df["Kappa"].mean()),
    "mIoU_mean": float(df["mIoU"].mean()),
    "wF1_mean": float(df["wF1"].mean()),
    "AUROC_mean": float(df["AUROC"].dropna().mean()) if "AUROC" in df.columns else float('nan'),
    "AP_mean": float(df["AP"].dropna().mean()) if "AP" in df.columns else float('nan'),
}
pd.DataFrame([summary]).to_csv(OUTPUT_DIR / "metrics_summary.csv", index=False)
print("Saved summary to", OUTPUT_DIR / "metrics_summary.csv")
print("Aggregated confusion matrix (total):\n", conf_total)

# ---------------- EDA plots ----------------
sns.set(style="whitegrid")
# 1) confusion matrix heatmap (normalized)
cm_total = conf_total.copy()
cm_norm = cm_total.astype(np.float32) / cm_total.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm_norm, annot=cm_total, fmt='d', cmap='Blues', ax=ax, cbar=False)
ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth"); ax.set_title("Confusion matrix (counts; normalized rows)")
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=200, bbox_inches='tight')
plt.close(fig)

# 2) per-class IoU bar
if "iou_c0" in df.columns:
    fig, ax = plt.subplots(figsize=(5,3))
    iou_mean = [df["iou_c0"].mean(), df["iou_c1"].mean()]
    sns.barplot(x=["class0","class1"], y=iou_mean, ax=ax)
    ax.set_ylim(0,1); ax.set_ylabel("IoU")
    ax.set_title("Per-class IoU (mean)")
    plt.savefig(OUTPUT_DIR / "per_class_iou.png", dpi=200, bbox_inches='tight')
    plt.close(fig)

# 3) weighted F1 / mIoU ranking
fig, ax = plt.subplots(figsize=(8,4))
df_sorted = df.sort_values("mIoU", ascending=False).reset_index(drop=True)
sns.barplot(x="file", y="mIoU", data=df_sorted.head(20), ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_title("Top 20 samples by mIoU")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "top20_mIoU.png", dpi=200)
plt.close(fig)

# 4) probability distribution and ROC/PR aggregate
# Concatenate all probs and gts
all_probs_flat = np.concatenate(all_probs) if len(all_probs)>0 else np.array([])
all_gts_flat = np.concatenate(all_gts) if len(all_gts)>0 else np.array([])

if all_probs_flat.size>0 and len(np.unique(all_gts_flat))>1:
    # ROC
    try:
        fpr, tpr, _ = roc_curve(all_gts_flat, all_probs_flat)
        fig, ax = plt.subplots(1,2, figsize=(10,4))
        ax[0].plot(fpr, tpr, label=f"AUC={roc_auc_score(all_gts_flat, all_probs_flat):.3f}")
        ax[0].plot([0,1],[0,1],'k--', alpha=0.5); ax[0].set_title("ROC Curve"); ax[0].legend()
        # PR curve
        prec, rec, _ = precision_recall_curve(all_gts_flat, all_probs_flat)
        ap = average_precision_score(all_gts_flat, all_probs_flat)
        ax[1].plot(rec, prec, label=f"AP={ap:.3f}"); ax[1].set_title("Precision-Recall")
        ax[1].legend()
        plt.savefig(OUTPUT_DIR / "roc_pr.png", dpi=200, bbox_inches='tight')
        plt.close(fig)
    except Exception as e:
        print("ROC/PR plotting failed:", e)

# 5) class distribution (pixel counts)
df['gt_pos_pixels'] = df['gt_pos_pixels'].astype(int)
df['pred_pos_pixels'] = df['pred_pos_pixels'].astype(int)
fig, ax = plt.subplots(figsize=(6,3))
sns.histplot(df['gt_pos_pixels'], label='gt pos pixels', color='C0', stat='count', binwidth=100, alpha=0.6)
sns.histplot(df['pred_pos_pixels'], label='pred pos pixels', color='C1', stat='count', binwidth=100, alpha=0.6)
ax.legend(); ax.set_title("Positive pixel counts per-sample (gt vs pred)")
plt.savefig(OUTPUT_DIR / "pos_pixel_counts.png", dpi=200, bbox_inches='tight')
plt.close(fig)

# 6) Save a small panel of example images (input, gt, prob heatmap, overlay)
SAMPLE_N = min(12, len(tif_files))
examples = random.sample(tif_files, SAMPLE_N)
from PIL import Image
fig_rows = min(6, SAMPLE_N)
fig_cols = 4
fig, axes = plt.subplots(fig_rows, fig_cols, figsize=(4*fig_cols, 3*fig_rows))
axes = axes.reshape(fig_rows, fig_cols)
for idx, f in enumerate(examples[:fig_rows]):
    img_resized, profile = read_tif_as_rgb(f, image_size=IMAGE_SIZE, percentile_clip=True)
    # preprocess + predict single
    t = preprocess_img_for_model(img_resized, processor=processor)
    with torch.no_grad():
        out = model(t.to(DEVICE))
        p = torch.softmax(out, dim=1).cpu().numpy()[0]
    prob_map = p[1] if p.shape[0] > 1 else p[0]
    pred_mask = (prob_map >= THRESHOLD).astype(np.uint8)
    gt_mask = extract_mask_from_arr(img_resized)
    # show
    r = idx
    axes[r,0].imshow(img_resized); axes[r,0].set_title("input"); axes[r,0].axis('off')
    axes[r,1].imshow(gt_mask, cmap='gray'); axes[r,1].set_title("gt"); axes[r,1].axis('off')
    axes[r,2].imshow(prob_map, cmap='inferno'); axes[r,2].set_title("probability (class1)"); axes[r,2].axis('off')
    overlay_rgb = (img_resized*255).astype(np.uint8).copy()
    overlay_rgb[pred_mask==1] = [255,0,0]
    axes[r,3].imshow(overlay_rgb); axes[r,3].set_title("pred overlay"); axes[r,3].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "examples_panel.png", dpi=200, bbox_inches='tight')
plt.close(fig)

print("EDA images saved to", OUTPUT_DIR)

# Print a short summary
print("SUMMARY:")
print(pd.DataFrame([summary]).T)
print("Done. Files saved in", OUTPUT_DIR)

# 3D UNet++

In [ ]:
# Fixed scheduler: 3D UNet++ enhanced (no verbose arg + LR logging)
import os, random
from pathlib import Path
import numpy as np
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import rasterio
import matplotlib.pyplot as plt
from transformers import AutoImageProcessor
from sklearn.metrics import confusion_matrix, accuracy_score, cohen_kappa_score
import pandas as pd

# ---------------- USER CONFIG ----------------
DATA_ROOT = Path('/kaggle/input/sentinel2-crop-mapping')
MAX_TILES_PER_REGION = 20
MAX_IMAGES_PER_TILE = 8
SEQ_LEN = 4
IMAGE_SIZE = 128
BATCH_SIZE = 4
EPOCHS = 6
NUM_WORKERS = 0
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BACKBONE = 'facebook/dinov2-small'
OUT_DIR = Path('output_3d_enhanced_fixed_scheduler'); OUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 42
BASE_CHANNELS = 12
LR = 3e-4
GRAD_CLIP = 1.0
PATIENCE = 2
WEIGHT_DECAY = 1e-4

# EXTRA FLAGS
NUM_CLASSES = 2        # set >2 to test multi-class pseudolabels (bins on band0)
USE_FOCAL = True       # if True use focal loss (multi-class), otherwise CrossEntropy
SWEEP = False          # set True to run a tiny hyperparameter sweep (WARNING: increases runtime)
# ------------------------------------------------

random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)

# processor for mean/std (optional)
try:
    processor = AutoImageProcessor.from_pretrained(BACKBONE)
except Exception:
    processor = None

# ---------------- helpers ----------------
def discover_regions_and_build_tile_map(root: Path, max_tiles_per_region=MAX_TILES_PER_REGION, max_images_per_tile=MAX_IMAGES_PER_TILE):
    region_dirs = []
    for d in root.iterdir():
        if not d.is_dir(): continue
        if any(child.is_dir() and child.name.startswith('data') for child in d.iterdir()):
            region_dirs.append(d.name)
    region_dirs = sorted(region_dirs)
    tile_map = {}
    for region in region_dirs:
        years = sorted([int(p.name.replace('data','')) for p in (root/region).iterdir() if p.is_dir() and p.name.startswith('data')])
        tileids = set()
        for y in years:
            base = root / region / f"data{y}"
            if not base.exists(): continue
            for t in base.iterdir():
                if t.is_dir(): tileids.add(t.name)
        tileids = sorted(tileids)
        chosen = random.sample(tileids, min(max_tiles_per_region, len(tileids))) if len(tileids)>0 else []
        for tileid in chosen:
            entries = []
            for y in years:
                tdir = root / region / f"data{y}" / tileid
                if not tdir.exists(): continue
                tifs = sorted([p.name for p in tdir.glob('*.tif')])
                for fn in tifs:
                    entries.append((region, int(y), tileid, fn))
                    if len(entries) >= max_images_per_tile: break
                if len(entries) >= max_images_per_tile: break
            if len(entries)>0:
                tile_map[(region, tileid)] = entries
    return tile_map, region_dirs

def read_image_entry(root: Path, region: str, year: int, tileid: str, fname: str, target_size=IMAGE_SIZE):
    p = root / region / f"data{year}" / tileid / fname
    if not p.exists():
        # fallback search
        found = None
        for child in (root / region).iterdir():
            if child.is_dir() and child.name.startswith('data'):
                t = child / tileid / fname
                if t.exists():
                    found = t; break
        if found is None:
            return None, None
        p = found
    try:
        with rasterio.open(p) as src:
            arr = src.read(out_dtype='float32')  # (C,H,W)
            img_arr = arr[:3] if arr.shape[0] >= 3 else np.vstack([arr]*3)[:3]
            img_arr = img_arr.astype('float32')
            img_arr = (img_arr - img_arr.min()) / (img_arr.max() - img_arr.min() + 1e-6)
            img_hwc = np.moveaxis(img_arr, 0, -1)
            pil = Image.fromarray((np.clip(img_hwc,0,1)*255).astype('uint8'))
            pil = pil.resize((target_size, target_size), resample=Image.BILINEAR)
            np_resized = np.asarray(pil).astype('float32') / 255.0
            return pil, np_resized
    except Exception:
        return None, None

# ---------------- dataset with multi-class pseudo-label option ----------------
class SentinelSeqDatasetMulti(Dataset):
    def __init__(self, sequences, root: Path, seq_len=SEQ_LEN, image_size=IMAGE_SIZE, processor=None, num_classes=NUM_CLASSES):
        self.sequences = sequences
        self.root = root
        self.seq_len = seq_len
        self.image_size = image_size
        self.processor = processor
        self.num_classes = num_classes
        pos = 0; tot = 0
        for seq in self.sequences[:min(len(self.sequences),200)]:
            region, year, tileid, fn = seq[-1]
            _, arr = read_image_entry(root, region, year, tileid, fn, target_size=self.image_size)
            if arr is None: continue
            band0 = arr[...,0]
            if self.num_classes == 2:
                mask = (band0 > band0.mean()).astype('int64')
                pos += mask.sum(); tot += mask.size
            else:
                flat = band0.ravel()
                if np.isclose(flat.max(), flat.min()):
                    mask = np.zeros_like(band0, dtype='int64')
                else:
                    thresholds = np.quantile(flat, q=np.linspace(0,1,self.num_classes+1)[1:-1])
                    mask = np.digitize(band0, bins=thresholds).astype('int64')
                pos += (mask>0).sum(); tot += mask.size
        self.pos = int(pos); self.tot = int(tot) if tot>0 else 1

    def __len__(self): return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        frames = []
        last_mask = None
        for i in range(self.seq_len):
            if i < len(seq):
                region, year, tileid, fname = seq[i]
                pil, arr = read_image_entry(self.root, region, year, tileid, fname, target_size=self.image_size)
                if arr is None:
                    frames.append(np.zeros((self.image_size, self.image_size,3), dtype='float32'))
                else:
                    frames.append(arr)
                if i == len(seq)-1 and arr is not None:
                    band0 = arr[...,0]
                    if self.num_classes == 2:
                        last_mask = (band0 > band0.mean()).astype('int64')
                    else:
                        flat = band0.ravel()
                        if np.isclose(flat.max(), flat.min()):
                            last_mask = np.zeros_like(band0, dtype='int64')
                        else:
                            thresholds = np.quantile(flat, q=np.linspace(0,1,self.num_classes+1)[1:-1])
                            last_mask = np.digitize(band0, bins=thresholds).astype('int64')
            else:
                frames.append(np.zeros((self.image_size, self.image_size,3), dtype='float32'))
        arr_stack = np.stack(frames, axis=0)  # D,H,W,C
        arr_stack = arr_stack.transpose(3,0,1,2).astype('float32')  # C,D,H,W
        x = torch.from_numpy(arr_stack)
        if self.processor is not None and hasattr(self.processor, "image_mean") and hasattr(self.processor, "image_std"):
            mean = torch.tensor(self.processor.image_mean, dtype=torch.float32).view(3,1,1,1)
            std  = torch.tensor(self.processor.image_std, dtype=torch.float32).view(3,1,1,1)
            x = (x - mean) / std
        else:
            x = (x - 0.5) / 0.5
        if last_mask is None:
            mask_np = np.zeros((self.image_size, self.image_size), dtype='int64')
        else:
            mask_np = last_mask
        mask_t = torch.from_numpy(mask_np).long()
        return x.float(), mask_t

# ---------------- build sequence list (correctly) ----------------
tile_map, regions = discover_regions_and_build_tile_map(DATA_ROOT, max_tiles_per_region=MAX_TILES_PER_REGION, max_images_per_tile=MAX_IMAGES_PER_TILE)
seq_samples = []
for (region, tileid), frames in tile_map.items():
    frames_list = frames
    # sliding windows over frames_list
    for i in range(0, max(1, len(frames_list))):
        seq = frames_list[i:i+SEQ_LEN]
        if seq:
            seq_samples.append(seq)
    if len(seq_samples) > 20000: break

random.shuffle(seq_samples)
split = int(0.8 * len(seq_samples))
train_seqs = seq_samples[:split]
val_seqs   = seq_samples[split: split + 400]

train_ds = SentinelSeqDatasetMulti(train_seqs, DATA_ROOT, seq_len=SEQ_LEN, image_size=IMAGE_SIZE, processor=processor, num_classes=NUM_CLASSES)
val_ds   = SentinelSeqDatasetMulti(val_seqs, DATA_ROOT, seq_len=SEQ_LEN, image_size=IMAGE_SIZE, processor=processor, num_classes=NUM_CLASSES)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("Train/Val counts:", len(train_ds), len(val_ds), "regions:", regions)
print("Estimated positive pixels (train):", train_ds.pos, "of", train_ds.tot, "NUM_CLASSES:", NUM_CLASSES)

# ---------------- 3D UNet++ architecture ----------------
def conv3d_block(in_ch, out_ch, kernel=3, padding=1, dropout=0.1):
    return nn.Sequential(
        nn.Conv3d(in_ch, out_ch, kernel_size=kernel, padding=padding),
        nn.BatchNorm3d(out_ch),
        nn.ReLU(inplace=True),
        nn.Dropout3d(dropout),
        nn.Conv3d(out_ch, out_ch, kernel_size=kernel, padding=padding),
        nn.BatchNorm3d(out_ch),
        nn.ReLU(inplace=True),
        nn.Dropout3d(dropout),
    )

class Up3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, out_ch, kernel_size=(1,2,2), stride=(1,2,2))
    def forward(self, x): return self.up(x)

class UNet3D_PP_Robust(nn.Module):
    def __init__(self, in_channels=3, base_ch=BASE_CHANNELS, num_classes=NUM_CLASSES, depth=3):
        super().__init__()
        self.pool = nn.MaxPool3d(kernel_size=(1,2,2), stride=(1,2,2))
        self.enc_blocks = nn.ModuleList()
        ch = in_channels
        for i in range(depth):
            outc = base_ch * (2**i)
            self.enc_blocks.append(conv3d_block(ch, outc))
            ch = outc
        self.bottleneck = conv3d_block(ch, ch*2)
        # decoder
        self.up_blocks = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        dec_in = ch*2
        for i in reversed(range(depth)):
            outc = base_ch * (2**i)
            self.up_blocks.append(Up3D(dec_in, outc))
            self.dec_convs.append(conv3d_block(outc*2, outc))
            dec_in = outc
        self.final_conv3d = nn.Conv3d(dec_in, num_classes, kernel_size=1)

    def forward(self, x):
        enc_feats = []
        out = x
        for enc in self.enc_blocks:
            out = enc(out)
            enc_feats.append(out)
            out = self.pool(out)
        out = self.bottleneck(out)
        for i, (up, dec) in enumerate(zip(self.up_blocks, self.dec_convs)):
            out = up(out)
            skip = enc_feats[-(i+1)]
            if out.shape[2:] != skip.shape[2:]:
                out = nn.functional.interpolate(out, size=skip.shape[2:], mode='trilinear', align_corners=False)
            out = torch.cat([out, skip], dim=1)
            out = dec(out)
        logits3d = self.final_conv3d(out)  # B, num_classes, D, H, W
        agg = logits3d.mean(dim=2)         # B, num_classes, H, W
        return agg

model = UNet3D_PP_Robust(in_channels=3, base_ch=BASE_CHANNELS, num_classes=NUM_CLASSES, depth=3).to(DEVICE)

# ---------------- loss functions ----------------
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.reduction = reduction
    def forward(self, inputs, targets):
        C = inputs.shape[1]
        logits = inputs.permute(0,2,3,1).reshape(-1, C)  # N, C
        targets_flat = targets.view(-1)                 # N
        logpt = -nn.functional.cross_entropy(logits, targets_flat, weight=self.weight, reduction='none', ignore_index=-100)
        pt = torch.exp(logpt)
        loss = -((1-pt)**self.gamma) * logpt
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss

# estimate class support & weights
support = np.zeros(NUM_CLASSES, dtype=np.int64)
for i in range(min(200, len(train_ds))):
    _, m = train_ds[i]
    vals, counts = np.unique(m.numpy().ravel(), return_counts=True)
    for v,c in zip(vals, counts):
        if 0 <= int(v) < NUM_CLASSES:
            support[int(v)] += int(c)
total_support = support.sum()
class_weights = np.ones(NUM_CLASSES, dtype=np.float32)
if total_support>0:
    class_weights = (total_support / (support + 1e-6)).astype(np.float32)
    class_weights = class_weights / class_weights.mean()
class_weights_torch = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print("Estimated class support:", support, "class_weights:", class_weights)

if USE_FOCAL:
    criterion = FocalLoss(gamma=2.0, weight=class_weights_torch, reduction='mean')
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights_torch)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# ====== FIX: create ReduceLROnPlateau without 'verbose' arg (older PyTorch) ======
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=PATIENCE)

# track current LR to detect reductions and log them
current_lr = optimizer.param_groups[0]['lr']
print(f"Starting LR = {current_lr:.6g}")

# ---------------- training loop with robust per-class metrics ----------------
def compute_metrics_from_conf(conf_mat):
    N = conf_mat.shape[0]
    support = conf_mat.sum(axis=1).astype(np.float64)
    total_support = support.sum()
    precision = np.zeros(N, dtype=np.float64)
    recall = np.zeros(N, dtype=np.float64)
    f1 = np.zeros(N, dtype=np.float64)
    iou = np.zeros(N, dtype=np.float64)
    for c in range(N):
        tp = conf_mat[c,c]
        fp = conf_mat[:,c].sum() - tp
        fn = conf_mat[c,:].sum() - tp
        pden = tp + fp
        rden = tp + fn
        iden = tp + fp + fn
        precision[c] = (tp / pden) if pden > 0 else 0.0
        recall[c] = (tp / rden) if rden > 0 else 0.0
        f1[c] = (2*precision[c]*recall[c] / (precision[c] + recall[c])) if (precision[c] + recall[c]) > 0 else 0.0
        iou[c] = (tp / iden) if iden > 0 else 0.0
    if total_support > 0:
        wP = float((precision * support).sum() / total_support)
        wR = float((recall * support).sum() / total_support)
        wF1 = float((f1 * support).sum() / total_support)
    else:
        wP = wR = wF1 = 0.0
    mIoU = float(iou.mean())
    return {
        'per_class_precision': precision, 'per_class_recall': recall, 'per_class_f1': f1, 'per_class_iou': iou,
        'support': support, 'wP': wP, 'wR': wR, 'wF1': wF1, 'mIoU': mIoU
    }

metrics_rows = []
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0; n_samples = 0
    for imgs, masks in train_loader:
        imgs = imgs.to(DEVICE); masks = masks.to(DEVICE)
        outputs = model(imgs)         # B, C, H, W
        loss = criterion(outputs, masks)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total_loss += loss.item() * imgs.size(0); n_samples += imgs.size(0)
    train_loss = total_loss / max(1, n_samples)

    # validation accumulate confusion matrix for NUM_CLASSES
    model.eval()
    conf = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    all_preds = []; all_gts = []
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.to(DEVICE); masks = masks.to(DEVICE)
            outs = model(imgs)   # B,C,H,W
            preds = outs.argmax(dim=1).cpu().numpy().ravel()
            gts = masks.cpu().numpy().ravel()
            gts_clamped = np.clip(gts, 0, NUM_CLASSES-1)
            preds_clamped = np.clip(preds, 0, NUM_CLASSES-1)
            cm = confusion_matrix(gts_clamped, preds_clamped, labels=list(range(NUM_CLASSES)))
            conf += cm
            all_preds.append(preds_clamped); all_gts.append(gts_clamped)
    if len(all_preds)>0:
        all_preds = np.concatenate(all_preds); all_gts = np.concatenate(all_gts)
    else:
        all_preds = np.array([]); all_gts = np.array([])

    OA = float(accuracy_score(all_gts, all_preds)) if all_gts.size>0 else 0.0
    Kappa = float(cohen_kappa_score(all_gts, all_preds)) if all_gts.size>0 else 0.0

    m = compute_metrics_from_conf(conf)
    wP = m['wP']; wR = m['wR']; wF1 = m['wF1']; mIoU = m['mIoU']
    per_iou = m['per_class_iou']; per_prec = m['per_class_precision']; per_rec = m['per_class_recall']; support = m['support']

    print(f"[E{epoch+1}/{EPOCHS}] train_loss={train_loss:.4f} Val OA={OA:.4f} Kappa={Kappa:.4f} mIoU={mIoU:.4f} wF1={wF1:.4f}")

    metrics_rows.append({
        'epoch': epoch+1, 'loss': train_loss, 'OA': OA, 'Kappa': Kappa, 'mIoU': mIoU,
        'wP': wP, 'wR': wR, 'wF1': wF1,
        'per_iou': ';'.join([f"{v:.4f}" for v in per_iou]),
        'per_prec': ';'.join([f"{v:.4f}" for v in per_prec]),
        'per_rec': ';'.join([f"{v:.4f}" for v in per_rec]),
        'support': ';'.join([str(int(s)) for s in support]),
        **{f'conf_{i}{j}': int(conf[i,j]) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES)}
    })

    # scheduler step (ReduceLROnPlateau expects a scalar)
    scheduler.step(mIoU)

    # log LR change if happened
    new_lr = optimizer.param_groups[0]['lr']
    if new_lr < current_lr:
        print(f"  LR reduced: {current_lr:.6g} -> {new_lr:.6g}")
        current_lr = new_lr

    torch.save({'epoch': epoch+1, 'model_state': model.state_dict(), 'opt': optimizer.state_dict()}, OUT_DIR / f'model_3d_enhanced_epoch{epoch+1}.pth')

# save metrics CSV
metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(OUT_DIR / 'metrics_3d_enhanced_fixed_scheduler.csv', index=False)
print("Saved metrics CSV to", OUT_DIR / 'metrics_3d_enhanced_fixed_scheduler.csv')

# ---------------- visualization ----------------
model.eval()
try:
    imgs, masks = next(iter(val_loader))
except StopIteration:
    print("No validation batch to visualize")
else:
    with torch.no_grad():
        outs = model(imgs.to(DEVICE))
        preds = outs.argmax(dim=1).cpu().numpy()
    nvis = min(4, imgs.shape[0])
    fig, axs = plt.subplots(nvis, 4, figsize=(12, 3*nvis))
    for i in range(nvis):
        last_frame = imgs[i].cpu().numpy()[:, -1, :, :].transpose(1,2,0)
        if processor is not None and hasattr(processor, "image_mean"):
            mean=np.array(processor.image_mean).reshape(1,1,3); std=np.array(processor.image_std).reshape(1,1,3)
            last_frame = (last_frame * std) + mean
        last_frame = np.clip(last_frame, 0, 1)
        axs[i,0].imshow(last_frame); axs[i,0].set_title('last input')
        axs[i,1].imshow(masks[i].cpu().numpy(), cmap='tab20', vmin=0, vmax=max(1,NUM_CLASSES-1)); axs[i,1].set_title('gt')
        axs[i,2].imshow(preds[i], cmap='tab20', vmin=0, vmax=max(1,NUM_CLASSES-1)); axs[i,2].set_title('pred')
        axs[i,3].imshow((preds[i] != masks[i].cpu().numpy()).astype('int'), cmap='gray'); axs[i,3].set_title('error')
        for j in range(4): axs[i,j].axis('off')
    plt.tight_layout()
    out_vis = OUT_DIR / 'val_predictions_3d_enhanced_fixed_scheduler.png'
    plt.savefig(out_vis, dpi=200)
    print("Saved visualization to", out_vis)

print("Done — outputs & metrics in", OUT_DIR, "NUM_CLASSES:", NUM_CLASSES, "USE_FOCAL:", USE_FOCAL)

In [ ]:
# ===============================================
# 📊 EDA + FULL EVALUATION — 3D UNet++ Robust
# ===============================================

import os
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import (
    classification_report,
    roc_curve,
    auc,
    precision_recall_curve,
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix
)

# 📂 1. Create the Evaluation Output Directory
eval_save_dir = OUT_DIR / "eda_evaluation"
eval_save_dir.mkdir(exist_ok=True)
print(f"📁 Created evaluation directory at: {eval_save_dir}")

# Ensure model is in evaluation mode
model.eval()

all_preds = []
all_gts = []
all_probs = []

# Initialize confusion matrix
conf = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)

print("⏳ Running inference on validation set for 3D UNet++...")
with torch.no_grad():
    for imgs, masks in val_loader:
        imgs = imgs.to(DEVICE)
        masks = masks.to(DEVICE)

        logits = model(imgs) # Output: (B, NUM_CLASSES, H, W)
        probs = torch.softmax(logits, dim=1)

        preds = probs.argmax(dim=1).cpu().numpy().ravel()
        gts = masks.cpu().numpy().ravel()

        # Clamp values to prevent out-of-bounds indexing in confusion matrix
        preds = np.clip(preds, 0, NUM_CLASSES-1)
        gts = np.clip(gts, 0, NUM_CLASSES-1)

        cm = confusion_matrix(gts, preds, labels=list(range(NUM_CLASSES)))
        conf += cm

        all_preds.append(preds)
        all_gts.append(gts)

        # If binary classification, save probabilities for the positive class
        if NUM_CLASSES == 2:
            all_probs.append(probs[:, 1, :, :].cpu().numpy().ravel())

# Flatten aggregated lists
all_preds = np.concatenate(all_preds)
all_gts = np.concatenate(all_gts)
if NUM_CLASSES == 2:
    all_probs = np.concatenate(all_probs)

# =========================
# 📈 CORE METRICS
# =========================

OA = accuracy_score(all_gts, all_preds)
Kappa = cohen_kappa_score(all_gts, all_preds)

# Call the compute_metrics_from_conf function already in your memory
metrics = compute_metrics_from_conf(conf) 
mIoU = metrics['mIoU']
wF1 = metrics['wF1']
class_report_str = classification_report(all_gts, all_preds, digits=4)

print("====================================")
print("📊 FINAL VALIDATION METRICS (3D UNet++)")
print("====================================")
print(f"Overall Accuracy (OA): {OA:.4f}")
print(f"Cohen's Kappa:         {Kappa:.4f}")
print(f"Mean IoU:              {mIoU:.4f}")
print(f"Weighted F1:           {wF1:.4f}")
print("====================================")
print("\nPer-class metrics:")
print(class_report_str)

# 💾 Save Text Metrics
metrics_file_path = eval_save_dir / "evaluation_metrics_3d_unet.txt"
with open(metrics_file_path, "w") as f:
    f.write("====================================\n")
    f.write("📊 FINAL VALIDATION METRICS (3D UNet++)\n")
    f.write("====================================\n")
    f.write(f"Overall Accuracy (OA): {OA:.4f}\n")
    f.write(f"Cohen's Kappa:         {Kappa:.4f}\n")
    f.write(f"Mean IoU:              {mIoU:.4f}\n")
    f.write(f"Weighted F1:           {wF1:.4f}\n")
    f.write("====================================\n\n")
    f.write("Per-class metrics:\n")
    f.write(class_report_str)

# =========================
# 📊 CONFUSION MATRIX PLOT
# =========================

plt.figure(figsize=(6,5))
sns.heatmap(conf, annot=True, fmt='d', cmap='Oranges') # Switched to Oranges to visually distinguish from previous models
plt.title("Confusion Matrix (3D UNet++)")
plt.xlabel("Predicted")
plt.ylabel("Ground Truth")
plt.tight_layout()
plt.savefig(eval_save_dir / "confusion_matrix.png", dpi=300, bbox_inches='tight')
plt.show()

# =========================
# 📊 CLASS DISTRIBUTION
# =========================

unique, counts = np.unique(all_gts, return_counts=True)

plt.figure(figsize=(6,4))
plt.bar(unique, counts, color='coral')
plt.title("Ground Truth Pixel Distribution")
plt.xlabel("Class")
plt.ylabel("Pixel Count")
plt.xticks(unique)
plt.tight_layout()
plt.savefig(eval_save_dir / "class_distribution.png", dpi=300, bbox_inches='tight')
plt.show()

# =========================
# 📊 PER-CLASS IoU & F1
# =========================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# IoU Subplot (Using correct dictionary keys for this script)
axes[0].bar(range(NUM_CLASSES), metrics['per_class_iou'], color='mediumseagreen')
axes[0].set_title("Per-Class IoU")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("IoU")
axes[0].set_xticks(range(NUM_CLASSES))

# F1 Subplot (Using correct dictionary keys for this script)
axes[1].bar(range(NUM_CLASSES), metrics['per_class_f1'], color='cornflowerblue')
axes[1].set_title("Per-Class F1 Score")
axes[1].set_xlabel("Class")
axes[1].set_ylabel("F1 Score")
axes[1].set_xticks(range(NUM_CLASSES))

plt.tight_layout()
plt.savefig(eval_save_dir / "per_class_iou_f1.png", dpi=300, bbox_inches='tight')
plt.show()

# =========================
# 📈 ROC + PR (Binary Only)
# =========================

if NUM_CLASSES == 2:
    fpr, tpr, _ = roc_curve(all_gts, all_probs)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.title("Receiver Operating Characteristic (ROC)")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right")
    plt.savefig(eval_save_dir / "roc_curve.png", dpi=300, bbox_inches='tight')
    plt.show()

    precision, recall, _ = precision_recall_curve(all_gts, all_probs)

    plt.figure(figsize=(6,5))
    plt.plot(recall, precision, color='purple', lw=2)
    plt.title("Precision-Recall Curve")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.savefig(eval_save_dir / "pr_curve.png", dpi=300, bbox_inches='tight')
    plt.show()

print(f"✅ EDA + Evaluation Complete! All assets saved to: {eval_save_dir}")

# Temporo-Spatial ViT variants

In [ ]:
# Patched TemporalViT training cell — robust metrics (no nan) + scheduler fix + small improvements
# Paste into a Jupyter cell and run.
import os, random
from pathlib import Path
import numpy as np
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import rasterio
import matplotlib.pyplot as plt
from transformers import AutoImageProcessor
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
import pandas as pd

# ---------------- USER CONFIG ----------------
DATA_ROOT = Path('/kaggle/input/sentinel2-crop-mapping')
IMAGE_SIZE = 128        # H,W
SEQ_LEN = 4             # temporal depth
PATCH_SIZE = 16         # spatial patch size
EMBED_DIM = 128         # token embedding dim (small)
NUM_LAYERS = 4          # transformer depth (small)
NUM_HEADS = 4
MLP_RATIO = 2.0
VARIANT = 'factorized'  # choose: 'factorized', 'joint', 'tubelet'
BATCH_SIZE = 8
EPOCHS = 6
NUM_WORKERS = 0
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT_DIR = Path('output_temporal_vit_fixed'); OUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 42

# Robust / extra options
NUM_CLASSES = 2        # change >2 if you want pseudo multi-class
USE_FOCAL = False      # if True, use focal loss
LR = 3e-4
GRAD_CLIP = 1.0
PATIENCE = 2
WEIGHT_DECAY = 1e-4
# ----------------------------------------------
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)

# optional HF processor for mean/std
try:
    processor = AutoImageProcessor.from_pretrained('facebook/dinov2-small')
except Exception:
    processor = None

# ---------------- Dataset (same logic as earlier, minimal) ----------------
class SentinelSeqDatasetSimple(Dataset):
    def __init__(self, seq_entries, data_root: Path, seq_len=SEQ_LEN, image_size=IMAGE_SIZE, processor=None, num_classes=NUM_CLASSES):
        self.seq_entries = seq_entries
        self.data_root = data_root
        self.seq_len = seq_len
        self.image_size = image_size
        self.processor = processor
        self.num_classes = num_classes

    def __len__(self):
        return len(self.seq_entries)

    def _read_entry(self, region, year, tileid, fname):
        p = self.data_root / region / f"data{year}" / tileid / fname
        if not p.exists():
            # fallback search
            for child in (self.data_root / region).iterdir():
                if child.is_dir() and child.name.startswith('data'):
                    t = child / tileid / fname
                    if t.exists():
                        p = t
                        break
            if not p.exists():
                return None, None
        try:
            with rasterio.open(p) as src:
                arr = src.read(out_dtype='float32')  # C,H,W
                if arr.shape[0] >= 3:
                    img_arr = arr[:3]
                else:
                    img_arr = np.vstack([arr]*3)[:3]
                img_arr = (img_arr - img_arr.min()) / (img_arr.max() - img_arr.min() + 1e-6)
                img_hwc = np.moveaxis(img_arr, 0, -1) # H,W,C
                pil = Image.fromarray((np.clip(img_hwc,0,1)*255).astype('uint8'))
                pil = pil.resize((self.image_size, self.image_size), resample=Image.BILINEAR)
                arr_resized = np.asarray(pil).astype('float32')/255.0
                return pil, arr_resized
        except Exception:
            return None, None

    def __getitem__(self, idx):
        seq = self.seq_entries[idx]
        frames = []
        last_mask = None
        for i in range(self.seq_len):
            if i < len(seq):
                region, year, tileid, fname = seq[i]
                pil, arr = self._read_entry(region, year, tileid, fname)
                if arr is None:
                    frames.append(np.zeros((self.image_size, self.image_size,3), dtype='float32'))
                else:
                    frames.append(arr)
                if i == len(seq)-1 and arr is not None:
                    band0 = arr[...,0]
                    if self.num_classes == 2:
                        last_mask = (band0 > band0.mean()).astype('int64')
                    else:
                        # equal-frequency bins
                        flat = band0.ravel()
                        if np.isclose(flat.max(), flat.min()):
                            last_mask = np.zeros_like(band0, dtype='int64')
                        else:
                            thresholds = np.quantile(flat, q=np.linspace(0,1,self.num_classes+1)[1:-1])
                            last_mask = np.digitize(band0, bins=thresholds).astype('int64')
            else:
                frames.append(np.zeros((self.image_size, self.image_size,3), dtype='float32'))
        stack = np.stack(frames, axis=0)  # D,H,W,C
        arr_stack = stack.transpose(3,0,1,2).astype('float32')  # C,D,H,W
        x = torch.from_numpy(arr_stack)
        # normalize
        if self.processor is not None and hasattr(self.processor, "image_mean") and hasattr(self.processor, "image_std"):
            mean = torch.tensor(self.processor.image_mean, dtype=torch.float32).view(3,1,1,1)
            std  = torch.tensor(self.processor.image_std, dtype=torch.float32).view(3,1,1,1)
            x = (x - mean) / std
        else:
            x = (x - 0.5) / 0.5
        if last_mask is None:
            mask = np.zeros((self.image_size, self.image_size), dtype='int64')
        else:
            mask = last_mask
        mask_t = torch.from_numpy(mask).long()
        return x.float(), mask_t

# ---------------- collect sequences (small sample) ----------------
def collect_sequences(data_root: Path, max_tiles_per_region=20, max_images_per_tile=6, seq_len=SEQ_LEN, max_seqs=1200):
    region_dirs = []
    for d in data_root.iterdir():
        if not d.is_dir(): continue
        if any(child.is_dir() and child.name.startswith('data') for child in d.iterdir()):
            region_dirs.append(d.name)
    region_dirs = sorted(region_dirs)
    seqs = []
    for region in region_dirs:
        years = sorted([int(p.name.replace('data','')) for p in (data_root/region).iterdir() if p.is_dir() and p.name.startswith('data')])
        tileids = set()
        for y in years:
            p = data_root/region/f"data{y}"
            if not p.exists(): continue
            for t in p.iterdir():
                if t.is_dir(): tileids.add(t.name)
        tileids = sorted(tileids)
        chosen = random.sample(tileids, min(max_tiles_per_region, len(tileids))) if len(tileids)>0 else []
        for tile in chosen:
            entries = []
            for y in years:
                tdir = data_root/region/f"data{y}"/tile
                if not tdir.exists(): continue
                tifs = sorted([p.name for p in tdir.glob("*.tif")])
                for fn in tifs:
                    entries.append((region, y, tile, fn))
                    if len(entries) >= max_images_per_tile: break
                if len(entries) >= max_images_per_tile: break
            if entries:
                for i in range(0, max(1, len(entries))):
                    seq = entries[i:i+seq_len]
                    if seq:
                        seqs.append(seq)
                    if len(seqs) >= max_seqs: break
            if len(seqs) >= max_seqs: break
        if len(seqs) >= max_seqs: break
    return seqs, region_dirs

seq_entries, regions = collect_sequences(DATA_ROOT, max_tiles_per_region=20, max_images_per_tile=6, seq_len=SEQ_LEN, max_seqs=1200)
random.shuffle(seq_entries)
split = int(0.8 * len(seq_entries))
train_seqs = seq_entries[:split]
val_seqs = seq_entries[split: split + 300]

train_ds = SentinelSeqDatasetSimple(train_seqs, DATA_ROOT, seq_len=SEQ_LEN, image_size=IMAGE_SIZE, processor=processor, num_classes=NUM_CLASSES)
val_ds   = SentinelSeqDatasetSimple(val_seqs, DATA_ROOT, seq_len=SEQ_LEN, image_size=IMAGE_SIZE, processor=processor, num_classes=NUM_CLASSES)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("Collected seqs:", len(seq_entries), "Train/Val:", len(train_ds), len(val_ds), "Regions:", regions)

# ---------------- Model (unchanged) ----------------
# (Uses the same TemporalViT implementation as before; pasted here for convenience)
class PatchEmbedPerFrame(nn.Module):
    def __init__(self, img_size=IMAGE_SIZE, patch_size=PATCH_SIZE, in_ch=3, embed_dim=EMBED_DIM):
        super().__init__()
        assert img_size % patch_size == 0
        self.patch_size = patch_size
        self.nh = img_size // patch_size
        self.nw = img_size // patch_size
        self.num_patches = self.nh * self.nw
        self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        B, C, D, H, W = x.shape
        x2 = x.permute(0,2,1,3,4).reshape(B*D, C, H, W)
        y = self.proj(x2)
        _, E, Hp, Wp = y.shape
        y = y.reshape(B, D, E, Hp*Wp).permute(0,1,3,2)  # B, D, Np, E
        return y

class FeedForward(nn.Module):
    def __init__(self, dim, mlp_ratio=MLP_RATIO, dropout=0.0):
        super().__init__()
        hidden = int(dim * mlp_ratio)
        self.net = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, dim), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)

class FactorizedBlock(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.temporal_attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)
        self.spatial_attn  = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)
        self.norm1_t = nn.LayerNorm(dim); self.norm1_s = nn.LayerNorm(dim); self.norm2 = nn.LayerNorm(dim)
        self.ffn = FeedForward(dim)
    def forward(self, x):
        B, D, Np, E = x.shape
        xt = x.permute(0,2,1,3).reshape(B*Np, D, E)
        xtn = self.norm1_t(xt)
        out_t, _ = self.temporal_attn(xtn, xtn, xtn)
        out_t = out_t + xt
        out_t = out_t.reshape(B, Np, D, E).permute(0,2,1,3)
        xs = out_t.reshape(B*D, Np, E)
        xsn = self.norm1_s(xs)
        out_s, _ = self.spatial_attn(xsn, xsn, xsn)
        out_s = out_s + xs
        out_s = out_s.reshape(B, D, Np, E)
        out_ffn = self.ffn(self.norm2(out_s))
        out = out_s + out_ffn
        return out

class JointBlock(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(dim); self.ffn = FeedForward(dim); self.norm2 = nn.LayerNorm(dim)
    def forward(self, x):
        B, D, Np, E = x.shape
        tokens = x.reshape(B, D*Np, E)
        t = self.norm1(tokens)
        out, _ = self.attn(t, t, t)
        out = out + tokens
        out = out + self.ffn(self.norm2(out))
        out = out.reshape(B, D, Np, E)
        return out

class TubeletBlock(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.spatial = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(dim); self.ffn = FeedForward(dim); self.norm2 = nn.LayerNorm(dim)
    def forward(self, x):
        xavg = x.mean(dim=1)
        t = self.norm(xavg)
        out, _ = self.spatial(t, t, t)
        out = out + xavg
        out = out + self.ffn(self.norm2(out))
        out = out.unsqueeze(1).expand(-1, x.shape[1], -1, -1)
        return out

class TemporalViT_Seg(nn.Module):
    def __init__(self, variant='factorized', img_size=IMAGE_SIZE, patch_size=PATCH_SIZE,
                 seq_len=SEQ_LEN, embed_dim=EMBED_DIM, depth=NUM_LAYERS, num_heads=NUM_HEADS, num_classes=NUM_CLASSES):
        super().__init__()
        self.variant = variant; self.seq_len = seq_len
        self.patch_embed = PatchEmbedPerFrame(img_size=img_size, patch_size=patch_size, in_ch=3, embed_dim=embed_dim)
        self.nh = img_size // patch_size; self.nw = self.nh
        self.num_patches = self.nh * self.nw
        self.pos_spatial = nn.Parameter(torch.zeros(1, 1, self.num_patches, embed_dim))
        self.pos_temporal = nn.Parameter(torch.zeros(1, seq_len, 1, embed_dim))
        self.dropout = nn.Dropout(0.0)
        blocks = []
        for _ in range(depth):
            if variant == 'factorized':
                blocks.append(FactorizedBlock(embed_dim, num_heads))
            elif variant == 'joint':
                blocks.append(JointBlock(embed_dim, num_heads))
            elif variant == 'tubelet':
                blocks.append(TubeletBlock(embed_dim, num_heads))
            else:
                raise ValueError("unknown variant")
        self.blocks = nn.ModuleList(blocks)
        self.norm = nn.LayerNorm(embed_dim)
        self.decoder_conv = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim//2, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(embed_dim//2, embed_dim//4, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(embed_dim//4, num_classes, kernel_size=1)
        )

    def forward(self, x):
        B = x.shape[0]
        tokens = self.patch_embed(x)  # B, D, Np, E
        tokens = tokens + self.pos_spatial + self.pos_temporal
        tokens = self.dropout(tokens)
        for blk in self.blocks:
            tokens = blk(tokens)
        tokens = self.norm(tokens)
        tokens_agg = tokens.mean(dim=1)  # B, Np, E
        E = tokens_agg.shape[-1]
        t = tokens_agg.permute(0,2,1).contiguous().view(B, E, self.nh, self.nw)
        logits_patch = self.decoder_conv(t)
        logits_full = nn.functional.interpolate(logits_patch, scale_factor=self.patch_embed.patch_size, mode='bilinear', align_corners=False)
        return logits_full  # B, num_classes, H, W

# instantiate
model = TemporalViT_Seg(variant=VARIANT, img_size=IMAGE_SIZE, patch_size=PATCH_SIZE,
                       seq_len=SEQ_LEN, embed_dim=EMBED_DIM, depth=NUM_LAYERS, num_heads=NUM_HEADS, num_classes=NUM_CLASSES).to(DEVICE)
print("Model variant:", VARIANT, "Params:", sum(p.numel() for p in model.parameters())/1e6, "M")

# ---------------- Loss / optimizer / weights ----------------
# estimate class support for weights
support = np.zeros(NUM_CLASSES, dtype=np.int64)
for i in range(min(200, len(train_ds))):
    _, m = train_ds[i]
    vals, counts = np.unique(m.numpy().ravel(), return_counts=True)
    for v,c in zip(vals, counts):
        if 0 <= int(v) < NUM_CLASSES:
            support[int(v)] += int(c)
total_support = support.sum()
class_weights = np.ones(NUM_CLASSES, dtype=np.float32)
if total_support > 0:
    class_weights = (total_support / (support + 1e-6)).astype(np.float32)
    class_weights = class_weights / class_weights.mean()
class_weights_torch = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print("class support:", support, "class_weights:", class_weights)

# focal loss optional
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, reduction='mean'):
        super().__init__()
        self.gamma = gamma; self.weight = weight; self.reduction = reduction
    def forward(self, inputs, targets):
        C = inputs.shape[1]
        logits = inputs.permute(0,2,3,1).reshape(-1, C)
        targets_flat = targets.view(-1)
        logpt = -nn.functional.cross_entropy(logits, targets_flat, weight=self.weight, reduction='none', ignore_index=-100)
        pt = torch.exp(logpt)
        loss = -((1-pt)**self.gamma) * logpt
        return loss.mean() if self.reduction=='mean' else loss.sum()

criterion = FocalLoss(gamma=2.0, weight=class_weights_torch) if USE_FOCAL else nn.CrossEntropyLoss(weight=class_weights_torch)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=PATIENCE)  # no verbose arg
current_lr = optimizer.param_groups[0]['lr']
print("Starting LR:", current_lr)

# ---------------- robust metric helpers ----------------
def compute_metrics_from_conf(conf_mat):
    N = conf_mat.shape[0]
    support = conf_mat.sum(axis=1).astype(np.float64)
    total = support.sum()
    precision = np.zeros(N, dtype=np.float64)
    recall = np.zeros(N, dtype=np.float64)
    f1 = np.zeros(N, dtype=np.float64)
    iou = np.zeros(N, dtype=np.float64)
    for c in range(N):
        tp = conf_mat[c,c]
        fp = conf_mat[:,c].sum() - tp
        fn = conf_mat[c,:].sum() - tp
        pden = tp + fp
        rden = tp + fn
        iden = tp + fp + fn
        precision[c] = (tp / pden) if pden > 0 else 0.0
        recall[c] = (tp / rden) if rden > 0 else 0.0
        f1[c] = (2*precision[c]*recall[c] / (precision[c] + recall[c])) if (precision[c] + recall[c]) > 0 else 0.0
        iou[c] = (tp / iden) if iden > 0 else 0.0
    if total > 0:
        wP = float((precision * support).sum() / total)
        wR = float((recall * support).sum() / total)
        wF1 = float((f1 * support).sum() / total)
    else:
        wP = wR = wF1 = 0.0
    mIoU = float(iou.mean())
    return {'precision': precision, 'recall': recall, 'f1': f1, 'iou': iou, 'support': support, 'wP': wP, 'wR': wR, 'wF1': wF1, 'mIoU': mIoU}

# ---------------- quick sanity forward ----------------
imgs, masks = next(iter(train_loader))
print("Batch shapes imgs", imgs.shape, "masks", masks.shape)
with torch.no_grad():
    out = model(imgs.to(DEVICE))
print("Model output", out.shape)

# ---------------- training loop (robust metrics) ----------------
metrics = []
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0; n = 0
    for imgs, masks in train_loader:
        imgs = imgs.to(DEVICE); masks = masks.to(DEVICE)
        logits = model(imgs)  # B,C,H,W
        loss = criterion(logits, masks)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total_loss += loss.item() * imgs.size(0); n += imgs.size(0)
    train_loss = total_loss / max(1,n)

    # validation accumulate confusion matrix
    model.eval()
    conf = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    all_preds = []; all_gts = []
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.to(DEVICE); masks = masks.to(DEVICE)
            logits = model(imgs)
            preds = logits.argmax(dim=1).cpu().numpy().ravel()
            gts = masks.cpu().numpy().ravel()
            # clamp labels
            preds = np.clip(preds, 0, NUM_CLASSES-1)
            gts = np.clip(gts, 0, NUM_CLASSES-1)
            cm = confusion_matrix(gts, preds, labels=list(range(NUM_CLASSES)))
            conf += cm
            all_preds.append(preds); all_gts.append(gts)
    if len(all_preds)>0:
        all_preds = np.concatenate(all_preds); all_gts = np.concatenate(all_gts)
    else:
        all_preds = np.array([]); all_gts = np.array([])

    OA = float(accuracy_score(all_gts, all_preds)) if all_gts.size>0 else 0.0
    try:
        Kappa = float(cohen_kappa_score(all_gts, all_preds)) if all_gts.size>0 else 0.0
        if np.isnan(Kappa) or np.isinf(Kappa):
            Kappa = 0.0
    except Exception:
        Kappa = 0.0

    m = compute_metrics_from_conf(conf)
    wP = m['wP']; wR = m['wR']; wF1 = m['wF1']; mIoU = m['mIoU']
    per_iou = m['iou']; per_prec = m['precision']; per_rec = m['recall']; support = m['support']

    print(f"Epoch {epoch+1}/{EPOCHS} train_loss={train_loss:.4f} Val OA={OA:.4f} Kappa={Kappa:.4f} mIoU={mIoU:.4f} wP={wP:.4f} wR={wR:.4f} wF1={wF1:.4f}")

    metrics.append({
        'epoch': epoch+1, 'loss': train_loss, 'OA': OA, 'Kappa': Kappa, 'mIoU': mIoU,
        'wP': wP, 'wR': wR, 'wF1': wF1,
        'per_iou': ';'.join([f"{v:.4f}" for v in per_iou]),
        'per_prec': ';'.join([f"{v:.4f}" for v in per_prec]),
        'per_rec': ';'.join([f"{v:.4f}" for v in per_rec]),
        'support': ';'.join([str(int(s)) for s in support]),
        **{f'conf_{i}{j}': int(conf[i,j]) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES)}
    })

    # scheduler step (use scalar mIoU)
    scheduler.step(mIoU)
    # log LR change
    new_lr = optimizer.param_groups[0]['lr']
    if new_lr < current_lr:
        print(f"  LR reduced: {current_lr:.6g} -> {new_lr:.6g}")
        current_lr = new_lr

    torch.save({'epoch': epoch+1, 'model_state': model.state_dict(), 'opt': optimizer.state_dict()}, OUT_DIR / f"temporalvit_{VARIANT}_epoch{epoch+1}.pth")

pd.DataFrame(metrics).to_csv(OUT_DIR / f"metrics_temporalvit_{VARIANT}_fixed.csv", index=False)
print("Saved metrics to", OUT_DIR)

# ---------------- visualization ----------------
model.eval()
try:
    imgs, masks = next(iter(val_loader))
except StopIteration:
    print("No val data")
else:
    with torch.no_grad():
        logits = model(imgs.to(DEVICE))
        preds = logits.argmax(dim=1).cpu().numpy()
    nvis = min(6, imgs.shape[0])
    fig, axs = plt.subplots(nvis, 3, figsize=(9,3*nvis))
    for i in range(nvis):
        last = imgs[i].cpu().numpy()[:, -1, :, :].transpose(1,2,0)
        if processor is not None and hasattr(processor,"image_mean"):
            mean=np.array(processor.image_mean).reshape(1,1,3); std=np.array(processor.image_std).reshape(1,1,3)
            last = (last * std) + mean
        last = np.clip(last, 0, 1)
        axs[i,0].imshow(last); axs[i,0].set_title('last input')
        axs[i,1].imshow(masks[i].cpu().numpy(), cmap='tab20', vmin=0, vmax=max(1,NUM_CLASSES-1)); axs[i,1].set_title('gt')
        axs[i,2].imshow(preds[i], cmap='tab20', vmin=0, vmax=max(1,NUM_CLASSES-1)); axs[i,2].set_title('pred')
        for j in range(3): axs[i,j].axis('off')
    plt.tight_layout(); plt.savefig(OUT_DIR / f"viz_temporalvit_{VARIANT}_fixed.png", dpi=200)
    print("Saved viz to", OUT_DIR / f"viz_temporalvit_{VARIANT}_fixed.png")

print("Done — outputs in", OUT_DIR)

In [ ]:
# ===============================================
# 📊 EDA + FULL EVALUATION — TemporalViT Model
# ===============================================

import os
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import (
    classification_report,
    roc_curve,
    auc,
    precision_recall_curve,
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix
)

# 📂 1. Create the Output Directory in Kaggle
save_dir = "/kaggle/working/output_eval_TemporoSpatialViT"
os.makedirs(save_dir, exist_ok=True)
print(f"📁 Created output directory at: {save_dir}")

model.eval()

all_preds = []
all_gts = []
all_probs = []

conf = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)

with torch.no_grad():
    for imgs, masks in val_loader:
        imgs = imgs.to(DEVICE)
        masks = masks.to(DEVICE)

        logits = model(imgs)
        probs = torch.softmax(logits, dim=1)

        preds = probs.argmax(dim=1).cpu().numpy().ravel()
        gts = masks.cpu().numpy().ravel()

        preds = np.clip(preds, 0, NUM_CLASSES-1)
        gts = np.clip(gts, 0, NUM_CLASSES-1)

        cm = confusion_matrix(gts, preds, labels=list(range(NUM_CLASSES)))
        conf += cm

        all_preds.append(preds)
        all_gts.append(gts)

        if NUM_CLASSES == 2:
            all_probs.append(probs[:,1,:,:].cpu().numpy().ravel())

# Flatten
all_preds = np.concatenate(all_preds)
all_gts = np.concatenate(all_gts)
if NUM_CLASSES == 2:
    all_probs = np.concatenate(all_probs)

# =========================
# 📈 CORE METRICS
# =========================

OA = accuracy_score(all_gts, all_preds)
Kappa = cohen_kappa_score(all_gts, all_preds)

metrics = compute_metrics_from_conf(conf) # Assuming this is defined elsewhere in your code
mIoU = metrics['mIoU']
wF1 = metrics['wF1']
class_report_str = classification_report(all_gts, all_preds, digits=4)

# Print to console
print("====================================")
print("📊 FINAL VALIDATION METRICS")
print("====================================")
print(f"Overall Accuracy (OA): {OA:.4f}")
print(f"Cohen's Kappa:         {Kappa:.4f}")
print(f"Mean IoU:              {mIoU:.4f}")
print(f"Weighted F1:           {wF1:.4f}")
print("====================================")
print("\nPer-class metrics:")
print(class_report_str)

# 💾 Save Metrics to a Text File
metrics_file_path = os.path.join(save_dir, "evaluation_metrics.txt")
with open(metrics_file_path, "w") as f:
    f.write("====================================\n")
    f.write("📊 FINAL VALIDATION METRICS\n")
    f.write("====================================\n")
    f.write(f"Overall Accuracy (OA): {OA:.4f}\n")
    f.write(f"Cohen's Kappa:         {Kappa:.4f}\n")
    f.write(f"Mean IoU:              {mIoU:.4f}\n")
    f.write(f"Weighted F1:           {wF1:.4f}\n")
    f.write("====================================\n\n")
    f.write("Per-class metrics:\n")
    f.write(class_report_str)
print(f"✅ Saved metrics to {metrics_file_path}")

# =========================
# 📊 CONFUSION MATRIX PLOT
# =========================

plt.figure(figsize=(6,5))
sns.heatmap(conf, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Ground Truth")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "confusion_matrix.png"), dpi=300, bbox_inches='tight')
plt.show()

# =========================
# 📊 CLASS DISTRIBUTION
# =========================

unique, counts = np.unique(all_gts, return_counts=True)

plt.figure(figsize=(6,4))
plt.bar(unique, counts)
plt.title("Ground Truth Pixel Distribution")
plt.xlabel("Class")
plt.ylabel("Pixel Count")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "class_distribution.png"), dpi=300, bbox_inches='tight')
plt.show()

# =========================
# 📊 PER-CLASS IoU
# =========================

plt.figure(figsize=(6,4))
plt.bar(range(NUM_CLASSES), metrics['iou'])
plt.title("Per-Class IoU")
plt.xlabel("Class")
plt.ylabel("IoU")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "per_class_iou.png"), dpi=300, bbox_inches='tight')
plt.show()

# =========================
# 📊 PER-CLASS F1
# =========================

plt.figure(figsize=(6,4))
plt.bar(range(NUM_CLASSES), metrics['f1'])
plt.title("Per-Class F1 Score")
plt.xlabel("Class")
plt.ylabel("F1")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "per_class_f1.png"), dpi=300, bbox_inches='tight')
plt.show()

# =========================
# 📈 ROC + PR (Binary Only)
# =========================

if NUM_CLASSES == 2:

    fpr, tpr, _ = roc_curve(all_gts, all_probs)
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr)
    plt.title(f"ROC Curve (AUC = {roc_auc:.4f})")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.savefig(os.path.join(save_dir, "roc_curve.png"), dpi=300, bbox_inches='tight')
    plt.show()

    precision, recall, _ = precision_recall_curve(all_gts, all_probs)

    plt.figure()
    plt.plot(recall, precision)
    plt.title("Precision-Recall Curve")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.savefig(os.path.join(save_dir, "pr_curve.png"), dpi=300, bbox_inches='tight')
    plt.show()

print("✅ EDA + Evaluation Complete and Saved to Kaggle Working Directory!")

# UNet + ConvLSTM

In [ ]:
# UNet + ConvLSTM (patched: robust metrics, focal loss, scheduler fix)
# Paste and run in a Jupyter cell.
import os, random
from pathlib import Path
import numpy as np
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import rasterio
import matplotlib.pyplot as plt
from transformers import AutoImageProcessor
from sklearn.metrics import confusion_matrix, accuracy_score, cohen_kappa_score
import pandas as pd

# ---------------- USER CONFIG ----------------
DATA_ROOT = Path('/kaggle/input/sentinel2-crop-mapping')
IMAGE_SIZE = 128       # H,W
SEQ_LEN = 4            # temporal depth
MAX_TILES_PER_REGION = 20
MAX_IMAGES_PER_TILE = 6
BATCH_SIZE = 6
EPOCHS = 6
NUM_WORKERS = 0
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT_DIR = Path('output_unet_convlstm_fixed'); OUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 42
BASE_CHANNELS = 16     # small UNet width
LR = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
PATIENCE = 2

# Robust extras
NUM_CLASSES = 2       # set >2 for pseudo multi-class
USE_FOCAL = False     # if True use focal loss (multi-class)
# ----------------------------------------------
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)

# optional HF processor for mean/std (not required)
try:
    processor = AutoImageProcessor.from_pretrained('facebook/dinov2-small')
except Exception:
    processor = None

# ---------------- helpers: discover dataset ----------------
def discover_regions(root: Path):
    regions = []
    for d in root.iterdir():
        if not d.is_dir(): continue
        if any(child.is_dir() and child.name.startswith('data') for child in d.iterdir()):
            regions.append(d.name)
    return sorted(regions)

def build_tile_map(root: Path, max_tiles_per_region=MAX_TILES_PER_REGION, max_images_per_tile=MAX_IMAGES_PER_TILE):
    tile_map = {}
    regions = discover_regions(root)
    for region in regions:
        years = sorted([int(p.name.replace('data','')) for p in (root/region).iterdir() if p.is_dir() and p.name.startswith('data')])
        tileids = set()
        for y in years:
            base = root/region/f"data{y}"
            if not base.exists(): continue
            for t in base.iterdir():
                if t.is_dir(): tileids.add(t.name)
        tileids = sorted(tileids)
        chosen = random.sample(tileids, min(max_tiles_per_region, len(tileids))) if len(tileids)>0 else []
        for tileid in chosen:
            entries=[]
            for y in years:
                tdir = root/region/f"data{y}"/tileid
                if not tdir.exists(): continue
                tifs = sorted([p.name for p in tdir.glob('*.tif')])
                for fn in tifs:
                    entries.append((region, int(y), tileid, fn))
                    if len(entries) >= max_images_per_tile: break
                if len(entries) >= max_images_per_tile: break
            if entries:
                tile_map[(region, tileid)] = entries
    return tile_map

def read_image_entry(root: Path, region: str, year: int, tileid: str, fname: str, target_size=IMAGE_SIZE):
    p = root / region / f"data{year}" / tileid / fname
    if not p.exists():
        # fallback search across data* folders
        found = None
        for child in (root/region).iterdir():
            if child.is_dir() and child.name.startswith('data'):
                t = child / tileid / fname
                if t.exists():
                    found = t; break
        if found is None:
            return None, None
        p = found
    try:
        with rasterio.open(p) as src:
            arr = src.read(out_dtype='float32')  # (C,H,W)
            img_arr = arr[:3] if arr.shape[0]>=3 else np.vstack([arr]*3)[:3]
            img_arr = img_arr.astype('float32')
            # per-image min-max normalization to 0..1
            img_arr = (img_arr - img_arr.min()) / (img_arr.max() - img_arr.min() + 1e-6)
            img_hwc = np.moveaxis(img_arr, 0, -1)
            pil = Image.fromarray((np.clip(img_hwc,0,1)*255).astype('uint8'))
            pil = pil.resize((target_size, target_size), resample=Image.BILINEAR)
            arr_resized = np.asarray(pil).astype('float32')/255.0
            return pil, arr_resized
    except Exception:
        return None, None

# ---------------- dataset with optional pseudo multi-class ----------------
class SentinelSeqDatasetMulti(Dataset):
    def __init__(self, sequences, root: Path, seq_len=SEQ_LEN, image_size=IMAGE_SIZE, processor=None, num_classes=NUM_CLASSES):
        self.sequences = sequences
        self.root = root
        self.seq_len = seq_len
        self.image_size = image_size
        self.processor = processor
        self.num_classes = num_classes
        # quick support estimate
        pos=0; tot=0
        for seq in self.sequences[:min(len(self.sequences),200)]:
            region, year, tileid, fn = seq[-1]
            _, arr = read_image_entry(root, region, year, tileid, fn, target_size=self.image_size)
            if arr is None: continue
            band0 = arr[...,0]
            if self.num_classes == 2:
                mask = (band0 > band0.mean()).astype('int64')
            else:
                flat = band0.ravel()
                if np.isclose(flat.max(), flat.min()):
                    mask = np.zeros_like(band0, dtype='int64')
                else:
                    thresholds = np.quantile(flat, q=np.linspace(0,1,self.num_classes+1)[1:-1])
                    mask = np.digitize(band0, bins=thresholds).astype('int64')
            pos += (mask>0).sum(); tot += mask.size
        self.pos = int(pos); self.tot = int(tot) if tot>0 else 1

    def __len__(self): return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        frames=[]
        last_mask=None
        for i in range(self.seq_len):
            if i < len(seq):
                region, year, tileid, fname = seq[i]
                pil, arr = read_image_entry(self.root, region, year, tileid, fname, target_size=self.image_size)
                if arr is None:
                    frames.append(np.zeros((self.image_size, self.image_size,3), dtype='float32'))
                else:
                    frames.append(arr)
                if i == len(seq)-1 and arr is not None:
                    band0 = arr[...,0]
                    if self.num_classes == 2:
                        last_mask = (band0 > band0.mean()).astype('int64')
                    else:
                        flat = band0.ravel()
                        if np.isclose(flat.max(), flat.min()):
                            last_mask = np.zeros_like(band0, dtype='int64')
                        else:
                            thresholds = np.quantile(flat, q=np.linspace(0,1,self.num_classes+1)[1:-1])
                            last_mask = np.digitize(band0, bins=thresholds).astype('int64')
            else:
                frames.append(np.zeros((self.image_size, self.image_size,3), dtype='float32'))
        arr_stack = np.stack(frames, axis=0)  # D,H,W,C
        arr_stack = arr_stack.transpose(3,0,1,2).astype('float32')  # C,D,H,W
        x = torch.from_numpy(arr_stack)
        # normalization
        if self.processor is not None and hasattr(self.processor, "image_mean") and hasattr(self.processor, "image_std"):
            mean = torch.tensor(self.processor.image_mean, dtype=torch.float32).view(3,1,1,1)
            std  = torch.tensor(self.processor.image_std, dtype=torch.float32).view(3,1,1,1)
            x = (x - mean) / std
        else:
            x = (x - 0.5) / 0.5
        if last_mask is None:
            mask_np = np.zeros((self.image_size, self.image_size), dtype='int64')
        else:
            mask_np = last_mask
        mask_t = torch.from_numpy(mask_np).long()
        return x.float(), mask_t

# ---------------- build sequence list ----------------
tile_map = build_tile_map(DATA_ROOT, max_tiles_per_region=MAX_TILES_PER_REGION, max_images_per_tile=MAX_IMAGES_PER_TILE)
seq_samples=[]
for (region, tileid), frames in tile_map.items():
    frames_list = frames
    for i in range(0, max(1, len(frames_list))):
        seq = frames_list[i:i+SEQ_LEN]
        if seq: seq_samples.append(seq)
    if len(seq_samples) > 20000: break

random.shuffle(seq_samples)
split = int(0.8 * len(seq_samples))
train_seqs = seq_samples[:split]
val_seqs = seq_samples[split: split + 400]

train_ds = SentinelSeqDatasetMulti(train_seqs, DATA_ROOT, seq_len=SEQ_LEN, image_size=IMAGE_SIZE, processor=processor, num_classes=NUM_CLASSES)
val_ds   = SentinelSeqDatasetMulti(val_seqs, DATA_ROOT, seq_len=SEQ_LEN, image_size=IMAGE_SIZE, processor=processor, num_classes=NUM_CLASSES)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("Train/Val:", len(train_ds), len(val_ds), "Estimated pos pixels:", train_ds.pos, "of", train_ds.tot, "NUM_CLASSES:", NUM_CLASSES)

# ---------------- model: UNet encoder/decoder + ConvLSTM (same structure) ----------------
def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )

class SmallUNetEncoder(nn.Module):
    def __init__(self, in_ch=3, base_ch=BASE_CHANNELS):
        super().__init__()
        self.enc1 = conv_block(in_ch, base_ch)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = conv_block(base_ch, base_ch*2)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = conv_block(base_ch*2, base_ch*4)
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = conv_block(base_ch*4, base_ch*8)
    def forward(self, x):
        e1 = self.enc1(x); p1=self.pool1(e1)
        e2 = self.enc2(p1); p2=self.pool2(e2)
        e3 = self.enc3(p2); p3=self.pool3(e3)
        b = self.bottleneck(p3)
        return e1,e2,e3,b

class SmallUNetDecoder(nn.Module):
    def __init__(self, base_ch=BASE_CHANNELS, num_classes=NUM_CLASSES):
        super().__init__()
        self.up3 = nn.ConvTranspose2d(base_ch*8, base_ch*4, kernel_size=2, stride=2)
        self.dec3 = conv_block(base_ch*8, base_ch*4)
        self.up2 = nn.ConvTranspose2d(base_ch*4, base_ch*2, kernel_size=2, stride=2)
        self.dec2 = conv_block(base_ch*4, base_ch*2)
        self.up1 = nn.ConvTranspose2d(base_ch*2, base_ch, kernel_size=2, stride=2)
        self.dec1 = conv_block(base_ch*2, base_ch)
        self.final = nn.Conv2d(base_ch, num_classes, kernel_size=1)
    def forward(self, bottleneck, e3, e2, e1):
        d3 = self.up3(bottleneck)
        if d3.shape != e3.shape: d3 = nn.functional.interpolate(d3, size=e3.shape[2:], mode='bilinear', align_corners=False)
        d3 = torch.cat([d3, e3], dim=1); d3 = self.dec3(d3)
        d2 = self.up2(d3)
        if d2.shape != e2.shape: d2 = nn.functional.interpolate(d2, size=e2.shape[2:], mode='bilinear', align_corners=False)
        d2 = torch.cat([d2, e2], dim=1); d2 = self.dec2(d2)
        d1 = self.up1(d2)
        if d1.shape != e1.shape: d1 = nn.functional.interpolate(d1, size=e1.shape[2:], mode='bilinear', align_corners=False)
        d1 = torch.cat([d1, e1], dim=1); d1 = self.dec1(d1)
        out = self.final(d1)
        out = nn.functional.interpolate(out, size=(IMAGE_SIZE, IMAGE_SIZE), mode='bilinear', align_corners=False)
        return out

# ConvLSTM cell & wrapper
class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3, bias=True):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(input_dim + hidden_dim, 4*hidden_dim, kernel_size, padding=padding, bias=bias)
        self.hidden_dim = hidden_dim
    def forward(self, x, h_cur):
        h_prev, c_prev = h_cur
        combined = torch.cat([x, h_prev], dim=1)
        conv_output = self.conv(combined)
        ci, cf, cg, co = torch.split(conv_output, self.hidden_dim, dim=1)
        i = torch.sigmoid(ci); f = torch.sigmoid(cf); g = torch.tanh(cg); o = torch.sigmoid(co)
        c_next = f * c_prev + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

class ConvLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        self.cell = ConvLSTMCell(input_dim, hidden_dim, kernel_size=kernel_size)
    def forward(self, x):  # x: B, seq, C, H, W
        B, seq, C, H, W = x.shape
        h = torch.zeros(B, self.cell.hidden_dim, H, W, device=x.device)
        c = torch.zeros(B, self.cell.hidden_dim, H, W, device=x.device)
        for t in range(seq):
            h, c = self.cell(x[:, t], (h, c))
        return h

class UNetConvLSTM(nn.Module):
    def __init__(self, in_ch=3, base_ch=BASE_CHANNELS, seq_len=SEQ_LEN, num_classes=NUM_CLASSES):
        super().__init__()
        self.seq_len = seq_len
        self.encoder = SmallUNetEncoder(in_ch=in_ch, base_ch=base_ch)
        self.convlstm = ConvLSTM(input_dim=base_ch*8, hidden_dim=base_ch*8, kernel_size=3)
        self.decoder = SmallUNetDecoder(base_ch=base_ch, num_classes=num_classes)
    def forward(self, x):
        # x: B, C, D, H, W
        B, C, D, H, W = x.shape
        bottlenecks=[]
        last_e1 = last_e2 = last_e3 = None
        for t in range(D):
            frame = x[:,:,t,:,:]           # B,C,H,W
            e1,e2,e3,b = self.encoder(frame)
            bottlenecks.append(b)
            if t == D-1:
                last_e1, last_e2, last_e3 = e1,e2,e3
        bb = torch.stack(bottlenecks, dim=1)  # B, D, Cb, h, w
        agg = self.convlstm(bb)               # B, Cb, h, w
        out = self.decoder(agg, last_e3, last_e2, last_e1)
        return out

# instantiate model
model = UNetConvLSTM(in_ch=3, base_ch=BASE_CHANNELS, seq_len=SEQ_LEN, num_classes=NUM_CLASSES).to(DEVICE)
print("Model params (M):", sum(p.numel() for p in model.parameters())/1e6)

# ---------------- loss & optimizer ----------------
# estimate support to compute class weights
support = np.zeros(NUM_CLASSES, dtype=np.int64)
for i in range(min(200, len(train_ds))):
    _, m = train_ds[i]
    vals, counts = np.unique(m.numpy().ravel(), return_counts=True)
    for v,c in zip(vals, counts):
        if 0 <= int(v) < NUM_CLASSES:
            support[int(v)] += int(c)
total_support = support.sum()
class_weights = np.ones(NUM_CLASSES, dtype=np.float32)
if total_support > 0:
    class_weights = (total_support / (support + 1e-6)).astype(np.float32)
    class_weights = class_weights / class_weights.mean()
class_weights_t = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print("class support:", support, "class_weights:", class_weights)

# focal loss (multi-class) implementation
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, reduction='mean'):
        super().__init__()
        self.gamma = gamma; self.weight = weight; self.reduction = reduction
    def forward(self, inputs, targets):
        # inputs: B,C,H,W ; targets: B,H,W
        C = inputs.shape[1]
        logits = inputs.permute(0,2,3,1).reshape(-1, C)
        targets_flat = targets.view(-1)
        logpt = -nn.functional.cross_entropy(logits, targets_flat, weight=self.weight, reduction='none', ignore_index=-100)
        pt = torch.exp(logpt)
        loss = -((1-pt)**self.gamma) * logpt
        return loss.mean() if self.reduction=='mean' else loss.sum()

if USE_FOCAL:
    criterion = FocalLoss(gamma=2.0, weight=class_weights_t)
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights_t)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
# ReduceLROnPlateau WITHOUT verbose to be compatible with older PyTorch
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=PATIENCE)
current_lr = optimizer.param_groups[0]['lr']
print("Starting LR:", current_lr)

# ---------------- metric helpers ----------------
def compute_metrics_from_conf(conf_mat):
    N = conf_mat.shape[0]
    support = conf_mat.sum(axis=1).astype(np.float64)
    total = support.sum()
    precision = np.zeros(N, dtype=np.float64); recall = np.zeros(N, dtype=np.float64)
    f1 = np.zeros(N, dtype=np.float64); iou = np.zeros(N, dtype=np.float64)
    for c in range(N):
        tp = conf_mat[c,c]
        fp = conf_mat[:,c].sum() - tp
        fn = conf_mat[c,:].sum() - tp
        pden = tp + fp; rden = tp + fn; iden = tp + fp + fn
        precision[c] = (tp / pden) if pden > 0 else 0.0
        recall[c] = (tp / rden) if rden > 0 else 0.0
        f1[c] = (2*precision[c]*recall[c] / (precision[c] + recall[c])) if (precision[c] + recall[c]) > 0 else 0.0
        iou[c] = (tp / iden) if iden > 0 else 0.0
    if total > 0:
        wP = float((precision * support).sum() / total)
        wR = float((recall * support).sum() / total)
        wF1 = float((f1 * support).sum() / total)
    else:
        wP = wR = wF1 = 0.0
    mIoU = float(iou.mean())
    return {'precision': precision, 'recall': recall, 'f1': f1, 'iou': iou, 'support': support, 'wP': wP, 'wR': wR, 'wF1': wF1, 'mIoU': mIoU}

# ---------------- training loop ----------------
metrics_rows=[]
for epoch in range(EPOCHS):
    model.train()
    total_loss=0.0; n=0
    for imgs, masks in train_loader:
        imgs = imgs.to(DEVICE); masks = masks.to(DEVICE)
        logits = model(imgs)              # B,C,H,W
        loss = criterion(logits, masks)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total_loss += loss.item() * imgs.size(0); n += imgs.size(0)
    train_loss = total_loss / max(1,n)

    # validation: accumulate confusion matrix
    model.eval()
    conf = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    all_preds=[]; all_gts=[]
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.to(DEVICE); masks = masks.to(DEVICE)
            logits = model(imgs)
            preds = logits.argmax(dim=1).cpu().numpy().ravel()
            gts = masks.cpu().numpy().ravel()
            preds_clamped = np.clip(preds, 0, NUM_CLASSES-1)
            gts_clamped = np.clip(gts, 0, NUM_CLASSES-1)
            cm = confusion_matrix(gts_clamped, preds_clamped, labels=list(range(NUM_CLASSES)))
            conf += cm
            all_preds.append(preds_clamped); all_gts.append(gts_clamped)
    if len(all_preds)>0:
        all_preds = np.concatenate(all_preds); all_gts = np.concatenate(all_gts)
    else:
        all_preds = np.array([]); all_gts = np.array([])

    OA = float(accuracy_score(all_gts, all_preds)) if all_gts.size>0 else 0.0
    try:
        Kappa = float(cohen_kappa_score(all_gts, all_preds)) if all_gts.size>0 else 0.0
        if np.isnan(Kappa) or np.isinf(Kappa): Kappa = 0.0
    except Exception:
        Kappa = 0.0

    m = compute_metrics_from_conf(conf)
    wP = m['wP']; wR = m['wR']; wF1 = m['wF1']; mIoU = m['mIoU']
    per_iou = m['iou']; per_prec = m['precision']; per_rec = m['recall']; support = m['support']

    print(f"[E{epoch+1}/{EPOCHS}] train_loss={train_loss:.4f} Val OA={OA:.4f} Kappa={Kappa:.4f} mIoU={mIoU:.4f} wP={wP:.4f} wR={wR:.4f} wF1={wF1:.4f}")

    metrics_rows.append({
        'epoch': epoch+1, 'loss': train_loss, 'OA': OA, 'Kappa': Kappa, 'mIoU': mIoU,
        'wP': wP, 'wR': wR, 'wF1': wF1,
        'per_iou': ';'.join([f"{v:.4f}" for v in per_iou]),
        'per_prec': ';'.join([f"{v:.4f}" for v in per_prec]),
        'per_rec': ';'.join([f"{v:.4f}" for v in per_rec]),
        'support': ';'.join([str(int(s)) for s in support]),
        **{f'conf_{i}{j}': int(conf[i,j]) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES)}
    })

    scheduler.step(mIoU)
    new_lr = optimizer.param_groups[0]['lr']
    if new_lr < current_lr:
        print(f"  LR reduced: {current_lr:.6g} -> {new_lr:.6g}")
        current_lr = new_lr

    torch.save({'epoch': epoch+1, 'model_state': model.state_dict(), 'opt': optimizer.state_dict()}, OUT_DIR / f'unet_convlstm_fixed_epoch{epoch+1}.pth')

# save metrics
pd.DataFrame(metrics_rows).to_csv(OUT_DIR / 'metrics_unet_convlstm_fixed.csv', index=False)
print("Saved metrics CSV to", OUT_DIR / 'metrics_unet_convlstm_fixed.csv')

# ---------------- visualization ----------------
model.eval()
try:
    imgs, masks = next(iter(val_loader))
except StopIteration:
    print("No validation batch to visualize")
else:
    with torch.no_grad():
        logits = model(imgs.to(DEVICE))
        preds = logits.argmax(dim=1).cpu().numpy()
    nvis = min(6, imgs.shape[0])
    fig, axs = plt.subplots(nvis, 3, figsize=(9, 3*nvis))
    for i in range(nvis):
        last = imgs[i].cpu().numpy()[:, -1, :, :].transpose(1,2,0)
        if processor is not None and hasattr(processor,"image_mean"):
            mean=np.array(processor.image_mean).reshape(1,1,3); std=np.array(processor.image_std).reshape(1,1,3)
            last = (last * std) + mean
        last = np.clip(last, 0, 1)
        axs[i,0].imshow(last); axs[i,0].set_title('last input')
        axs[i,1].imshow(masks[i].cpu().numpy(), cmap='tab20', vmin=0, vmax=max(1,NUM_CLASSES-1)); axs[i,1].set_title('gt')
        axs[i,2].imshow(preds[i], cmap='tab20', vmin=0, vmax=max(1,NUM_CLASSES-1)); axs[i,2].set_title('pred')
        for j in range(3): axs[i,j].axis('off')
    plt.tight_layout()
    out_vis = OUT_DIR / 'viz_unet_convlstm_fixed.png'
    plt.savefig(out_vis, dpi=200)
    print("Saved viz to", out_vis)

print("Done. Outputs saved to", OUT_DIR)

In [ ]:
# ===============================================
# 📊 EDA + FULL EVALUATION — UNet + ConvLSTM
# ===============================================

import os
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import (
    classification_report,
    roc_curve,
    auc,
    precision_recall_curve,
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix
)

# 📂 1. Create the Evaluation Output Directory
eval_save_dir = OUT_DIR / "eda_evaluation"
eval_save_dir.mkdir(exist_ok=True)
print(f"📁 Created evaluation directory at: {eval_save_dir}")

# Ensure model is in evaluation mode
model.eval()

all_preds = []
all_gts = []
all_probs = []

# Initialize confusion matrix
conf = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)

print("⏳ Running inference on validation set...")
with torch.no_grad():
    for imgs, masks in val_loader:
        imgs = imgs.to(DEVICE)
        masks = masks.to(DEVICE)

        logits = model(imgs) # Output: (B, NUM_CLASSES, H, W)
        probs = torch.softmax(logits, dim=1)

        preds = probs.argmax(dim=1).cpu().numpy().ravel()
        gts = masks.cpu().numpy().ravel()

        # Clamp values just in case
        preds = np.clip(preds, 0, NUM_CLASSES-1)
        gts = np.clip(gts, 0, NUM_CLASSES-1)

        cm = confusion_matrix(gts, preds, labels=list(range(NUM_CLASSES)))
        conf += cm

        all_preds.append(preds)
        all_gts.append(gts)

        # If binary classification, save probabilities for the positive class (class 1)
        if NUM_CLASSES == 2:
            all_probs.append(probs[:, 1, :, :].cpu().numpy().ravel())

# Flatten aggregated lists
all_preds = np.concatenate(all_preds)
all_gts = np.concatenate(all_gts)
if NUM_CLASSES == 2:
    all_probs = np.concatenate(all_probs)

# =========================
# 📈 CORE METRICS
# =========================

OA = accuracy_score(all_gts, all_preds)
Kappa = cohen_kappa_score(all_gts, all_preds)

# Re-using your compute_metrics_from_conf function from the training cell
metrics = compute_metrics_from_conf(conf) 
mIoU = metrics['mIoU']
wF1 = metrics['wF1']
class_report_str = classification_report(all_gts, all_preds, digits=4)

print("====================================")
print("📊 FINAL VALIDATION METRICS")
print("====================================")
print(f"Overall Accuracy (OA): {OA:.4f}")
print(f"Cohen's Kappa:         {Kappa:.4f}")
print(f"Mean IoU:              {mIoU:.4f}")
print(f"Weighted F1:           {wF1:.4f}")
print("====================================")
print("\nPer-class metrics:")
print(class_report_str)

# 💾 Save Text Metrics
metrics_file_path = eval_save_dir / "evaluation_metrics.txt"
with open(metrics_file_path, "w") as f:
    f.write("====================================\n")
    f.write("📊 FINAL VALIDATION METRICS\n")
    f.write("====================================\n")
    f.write(f"Overall Accuracy (OA): {OA:.4f}\n")
    f.write(f"Cohen's Kappa:         {Kappa:.4f}\n")
    f.write(f"Mean IoU:              {mIoU:.4f}\n")
    f.write(f"Weighted F1:           {wF1:.4f}\n")
    f.write("====================================\n\n")
    f.write("Per-class metrics:\n")
    f.write(class_report_str)

# =========================
# 📊 CONFUSION MATRIX PLOT
# =========================

plt.figure(figsize=(6,5))
sns.heatmap(conf, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix (UNet + ConvLSTM)")
plt.xlabel("Predicted")
plt.ylabel("Ground Truth")
plt.tight_layout()
plt.savefig(eval_save_dir / "confusion_matrix.png", dpi=300, bbox_inches='tight')
plt.show()

# =========================
# 📊 CLASS DISTRIBUTION
# =========================

unique, counts = np.unique(all_gts, return_counts=True)

plt.figure(figsize=(6,4))
plt.bar(unique, counts, color='skyblue')
plt.title("Ground Truth Pixel Distribution")
plt.xlabel("Class")
plt.ylabel("Pixel Count")
plt.xticks(unique)
plt.tight_layout()
plt.savefig(eval_save_dir / "class_distribution.png", dpi=300, bbox_inches='tight')
plt.show()

# =========================
# 📊 PER-CLASS IoU & F1
# =========================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# IoU Subplot
axes[0].bar(range(NUM_CLASSES), metrics['iou'], color='lightgreen')
axes[0].set_title("Per-Class IoU")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("IoU")
axes[0].set_xticks(range(NUM_CLASSES))

# F1 Subplot
axes[1].bar(range(NUM_CLASSES), metrics['f1'], color='salmon')
axes[1].set_title("Per-Class F1 Score")
axes[1].set_xlabel("Class")
axes[1].set_ylabel("F1 Score")
axes[1].set_xticks(range(NUM_CLASSES))

plt.tight_layout()
plt.savefig(eval_save_dir / "per_class_iou_f1.png", dpi=300, bbox_inches='tight')
plt.show()

# =========================
# 📈 ROC + PR (Binary Only)
# =========================

if NUM_CLASSES == 2:
    fpr, tpr, _ = roc_curve(all_gts, all_probs)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.title("Receiver Operating Characteristic (ROC)")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right")
    plt.savefig(eval_save_dir / "roc_curve.png", dpi=300, bbox_inches='tight')
    plt.show()

    precision, recall, _ = precision_recall_curve(all_gts, all_probs)

    plt.figure(figsize=(6,5))
    plt.plot(recall, precision, color='purple', lw=2)
    plt.title("Precision-Recall Curve")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.savefig(eval_save_dir / "pr_curve.png", dpi=300, bbox_inches='tight')
    plt.show()

print(f"✅ EDA + Evaluation Complete! All assets saved to: {eval_save_dir}")

In [ ]:
# Comparison table + plots for the 4 models
# Paste into a Jupyter cell and run. Requires pandas, matplotlib, glob, pathlib.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import glob
import os

# ----------------- Config: candidate metric file locations per model -----------------
candidates = {
    "DINOv2_2D": [
        "output/metrics.csv",
        "output/metrics_dinov2.csv",
        "metrics.csv",
    ],
    "3D_UNetPP_fixed": [
        "output_3d_enhanced_fixed_scheduler/metrics_3d_enhanced_fixed_scheduler.csv",
        "output_3d/metrics_3dunetpp.csv",
        "output_3d/metrics_3dunetpp.csv",
        "metrics_3d_fixed.csv",
        "metrics_3dunetpp.csv",
    ],
    "TemporalViT": [
        "output_temporal_vit_fixed/metrics_temporalvit_factorized_fixed.csv",
        "output_temporal_vit/metrics_temporalvit_tubelet.csv",
        "output_temporal_vit/metrics_temporalvit_joint.csv",
        "output_temporal_vit/metrics_temporalvit_factorized.csv",
        "metrics_temporalvit_*.csv",
    ],
    "UNet_ConvLSTM": [
        "output_unet_convlstm_fixed/metrics_unet_convlstm_fixed.csv",
        "metrics_unet_convlstm.csv",
    ]
}

# additional generic search locations (scan working dir for metrics files)
search_globs = [
    "output*/metrics*.csv",
    "metrics*.csv",
    "**/metrics*.csv",
]

# a helper to find first existing path from candidates; supports glob patterns
def find_metrics_file(patterns):
    for p in patterns:
        # expand user & make Path-like
        if "*" in p:
            found = glob.glob(p, recursive=True)
            if found:
                return Path(found[0])
        else:
            pth = Path(p)
            if pth.exists():
                return pth
            # try under working dir /kaggle/working
            alt = Path("/kaggle/working") / p
            if alt.exists():
                return alt
            # try relative recursive search
            found = list(Path(".").rglob(pth.name))
            if found:
                return found[0]
    # fallback: search generic globs
    for g in search_globs:
        found = glob.glob(g, recursive=True)
        if found:
            # try to choose the best match (prefer ones in output folders)
            # sort by path length (shorter first) then lexicographically
            found_sorted = sorted(found, key=lambda x: (len(x), x))
            return Path(found_sorted[0])
    return None

# ----------------- Read and summarize metrics -----------------
summary_rows = []
missing = {}
for model_name, patterns in candidates.items():
    mfile = find_metrics_file(patterns)
    if mfile is None:
        missing[model_name] = True
        print(f"[WARN] No metrics file found automatically for {model_name}. Searched patterns: {patterns}")
        continue
    try:
        df = pd.read_csv(mfile)
    except Exception as e:
        print(f"[ERR] Could not read {mfile} for {model_name}: {e}")
        missing[model_name] = True
        continue

    # standardize column names to lowercase
    df.columns = [c.strip() for c in df.columns]
    # columns we expect: 'epoch','loss','OA' or 'acc','Kappa','mIoU','wP','wR','wF1' etc.
    # normalize known names
    colmap = {}
    for c in df.columns:
        lc = c.lower()
        if lc in ("oa","accuracy","acc"):
            colmap[c] = "OA"
        elif lc in ("kappa","cohen_kappa","cohen_kappa_score"):
            colmap[c] = "Kappa"
        elif lc in ("miou","m_iou","mean_iou"):
            colmap[c] = "mIoU"
        elif lc in ("wp","precision_weighted","w_precision"):
            colmap[c] = "wP"
        elif lc in ("wr","recall_weighted","w_recall"):
            colmap[c] = "wR"
        elif lc in ("wf1","f1_weighted","w_f1","wf1_score"):
            colmap[c] = "wF1"
        elif 'conf00' in lc or 'conf_00' in lc or 'tn' in lc:
            colmap[c] = "conf00"
        elif 'conf01' in lc or 'conf_01' in lc:
            colmap[c] = "conf01"
        elif 'conf10' in lc or 'conf_10' in lc:
            colmap[c] = "conf10"
        elif 'conf11' in lc or 'conf_11' in lc or 'tp' in lc:
            colmap[c] = "conf11"
        elif lc in ("loss",):
            colmap[c] = "loss"
        elif lc in ("epoch",):
            colmap[c] = "epoch"
    df = df.rename(columns=colmap)

    # choose the row we treat as 'best' - prefer highest mIoU, fallback to OA, else lowest loss
    chosen_row = None
    if "mIoU" in df.columns and df["mIoU"].notna().any():
        idx = df["mIoU"].idxmax()
        chosen_row = df.loc[idx]
        chosen_metric = "mIoU"
    elif "OA" in df.columns and df["OA"].notna().any():
        idx = df["OA"].idxmax()
        chosen_row = df.loc[idx]
        chosen_metric = "OA"
    elif "loss" in df.columns and df["loss"].notna().any():
        idx = df["loss"].idxmin()
        chosen_row = df.loc[idx]
        chosen_metric = "loss"
    else:
        # take last row
        chosen_row = df.iloc[-1]
        chosen_metric = "last_row"

    # extract metrics (safely)
    def get_val(d, key):
        if key in d and pd.notna(d[key]):
            return float(d[key])
        return np.nan

    row = {
        "model": model_name,
        "metrics_file": str(mfile),
        "best_epoch": int(get_val(chosen_row, "epoch")) if not pd.isna(get_val(chosen_row, "epoch")) else None,
        "best_by": chosen_metric,
        "loss": get_val(chosen_row, "loss"),
        "OA": get_val(chosen_row, "OA"),
        "Kappa": get_val(chosen_row, "Kappa"),
        "mIoU": get_val(chosen_row, "mIoU"),
        "wP": get_val(chosen_row, "wP"),
        "wR": get_val(chosen_row, "wR"),
        "wF1": get_val(chosen_row, "wF1"),
        "conf00": int(get_val(chosen_row, "conf00")) if not np.isnan(get_val(chosen_row, "conf00")) else None,
        "conf01": int(get_val(chosen_row, "conf01")) if not np.isnan(get_val(chosen_row, "conf01")) else None,
        "conf10": int(get_val(chosen_row, "conf10")) if not np.isnan(get_val(chosen_row, "conf10")) else None,
        "conf11": int(get_val(chosen_row, "conf11")) if not np.isnan(get_val(chosen_row, "conf11")) else None,
    }
    summary_rows.append(row)

# ----------------- Build table -----------------
if len(summary_rows) == 0:
    raise RuntimeError("No metrics files found for any model. Make sure experiments finished and metrics CSVs are present.")

comp_df = pd.DataFrame(summary_rows)
# reorder columns nicely
cols_order = ["model", "metrics_file", "best_epoch", "best_by", "loss", "OA", "Kappa", "mIoU", "wP", "wR", "wF1",
              "conf00","conf01","conf10","conf11"]
comp_df = comp_df[ [c for c in cols_order if c in comp_df.columns] ]

# format numeric columns with 4 decimal places for display
pd.set_option("display.float_format", "{:0.4f}".format)
print("\n=== Comparison table (best epoch per model chosen by mIoU -> OA -> loss) ===\n")
display(comp_df)

# Save CSV
comp_out = Path("comparison_metrics.csv")
comp_df.to_csv(comp_out, index=False)
print("Saved comparison CSV to", comp_out)

# ----------------- Simple plots -----------------
# bar chart for mIoU and wF1 (handles missing values)
plot_df = comp_df.set_index("model")
plt.figure(figsize=(10,4))
if "mIoU" in plot_df.columns:
    plt.subplot(1,2,1)
    plot_df["mIoU"].plot(kind="bar", title="mIoU (best epoch)", ylim=(0,1))
    plt.ylabel("mIoU")
if "wF1" in plot_df.columns:
    plt.subplot(1,2,2)
    plot_df["wF1"].plot(kind="bar", title="weighted F1 (best epoch)", ylim=(0,1))
    plt.ylabel("wF1")
plt.tight_layout()
plt.savefig("comparison_mIoU_wF1.png", dpi=200)
print("Saved comparison plot to comparison_mIoU_wF1.png")
plt.show()

# Also print a short ranking by mIoU
if "mIoU" in comp_df.columns:
    ranking = comp_df[["model","mIoU"]].sort_values("mIoU", ascending=False)
    print("\nRanking by mIoU (descending):")
    display(ranking)

# report missing models
if missing:
    print("\nWARN: The following models had no metrics file found (or failed to read):")
    for k in missing:
        print("  -", k)

print("\nDone. If you want I can (A) compute per-class IoU if confusion matrix counts are present, (B) include epoch-by-epoch plots, or (C) automatically run the models sequentially and produce the metrics (takes time). Which next?")